# FDS Challenge: Pokémon Battle Prediction — final model

Classical machine learning only (no neural networks). Given the first 30 turns of a Gen 2 OU battle,
the notebook predicts whether player 1 wins.

**Pipeline**

1. **Features:** about 280 battle-state features from a Gen 2 damage engine, plus bag-of-words of
   Pokémon and moves.
2. **Monte Carlo features:** 100 simulated continuations of every battle.
3. **Recursive features:** turn-by-turn roll-outs of a gradient-boosted transition model.
4. **Snapshot augmentation:** training battles truncated at turns 24/26/28 are added inside the
   training folds, with weight 0.5.
5. **Stack of 10 diverse base models:** LightGBM, XGBoost and CatBoost with augmentation,
   ExtraTrees, Random Forest, LDA and two logistic regressions. A logistic-regression meta-learner
   combines their out-of-fold logits.

**Expected accuracy:** about 0.696 in nested cross-validation (0.699 with plain 5-fold OOF).

**Settings:**
- Accelerator: **None**. The CPU run takes about 30–40 minutes on Kaggle's 4 cores. With a GPU,
  XGBoost and CatBoost switch to CUDA and the predictions change slightly.
- Internet: not needed.

Source code and full documentation: https://github.com/MarcoValerioQuinti/pokemon-fds

### 0. Setup

In [ ]:
import glob
import os
import subprocess
import sys
import time

import pandas as pd


def find_data_dir():
    """Competition data: mounted under /kaggle/input on Kaggle, POKEMON_DATA or ./data elsewhere."""
    hits = sorted(glob.glob("/kaggle/input/**/train.jsonl", recursive=True))
    if hits:
        return os.path.dirname(hits[0])
    return os.environ.get("POKEMON_DATA", os.path.abspath("data"))


DATA_DIR = find_data_dir()
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
WORK_DIR = os.path.join(OUT_DIR, "outputs")  # feature caches, simulations, out-of-fold predictions
os.environ.update(POKEMON_DATA=DATA_DIR, POKEMON_OUT=WORK_DIR)
os.makedirs("src", exist_ok=True)

import catboost, lightgbm, sklearn, xgboost
print(f"data: {DATA_DIR} -> {sorted(os.listdir(DATA_DIR))}")
print(f"outputs: {WORK_DIR}; cores: {os.cpu_count()}")
print(f"scikit-learn {sklearn.__version__}, lightgbm {lightgbm.__version__}, "
      f"xgboost {xgboost.__version__}, catboost {catboost.__version__}")

### 1. Source code

Each cell writes one module to `src/`. The modules are copied verbatim from the repository; comments are in Italian.

**`src/compute.py`**: Hardware configuration: cores, CUDA, fork/spawn, fold-level parallelism.

In [ ]:
%%writefile src/compute.py
"""Configurazione hardware portabile (macOS / Linux / Windows).

Tutto è sovrascrivibile da variabili d'ambiente o dalle opzioni --n-jobs / --device degli script:
    POKEMON_N_JOBS   numero di processi/thread (default: tutti i core)
    POKEMON_DEVICE   auto | cpu | cuda (default: auto -> cuda se c'è una GPU NVIDIA)
    POKEMON_MP       fork | spawn (default: fork dove esiste, spawn su Windows)

- Multiprocessing: `fork` su Linux/macOS (i worker ereditano i dati del padre), `spawn` su
  Windows (dove `fork` non esiste); i dati passano dall'initializer, quindi funziona con entrambi.
- GPU: XGBoost (`device="cuda"`) e CatBoost (`task_type="GPU"`) solo con CUDA (Linux/Windows
  con NVIDIA). Su Mac Apple Silicon non c'è backend GPU per questi modelli: si usa la CPU.
- LightGBM resta su CPU (la build GPU non è quella standard di pip/conda).
"""
import multiprocessing as mp
import os
import shutil
import subprocess
import sys

_DEVICE = None


def n_jobs():
    return int(os.environ.get("POKEMON_N_JOBS", 0)) or os.cpu_count() or 1


def mp_context():
    forced = os.environ.get("POKEMON_MP")  # fork | spawn, per testare il percorso Windows altrove
    if forced:
        return mp.get_context(forced)
    methods = mp.get_all_start_methods()
    return mp.get_context("fork" if "fork" in methods and sys.platform != "win32" else "spawn")


def _has_cuda():
    if sys.platform == "darwin" or not shutil.which("nvidia-smi"):
        return False
    try:
        return subprocess.run(["nvidia-smi", "-L"], capture_output=True, timeout=10).returncode == 0
    except (OSError, subprocess.TimeoutExpired):
        return False


def device():
    global _DEVICE
    if _DEVICE is None:
        want = os.environ.get("POKEMON_DEVICE", "auto")
        _DEVICE = ("cuda" if _has_cuda() else "cpu") if want == "auto" else want
    return _DEVICE


def configure(n=None, dev=None):
    """Applica le opzioni da riga di comando (prima di creare modelli o pool)."""
    global _DEVICE
    if n:
        os.environ["POKEMON_N_JOBS"] = str(n)
    if dev:
        os.environ["POKEMON_DEVICE"] = dev
        _DEVICE = None


def xgb_params():
    return {"device": "cuda", "tree_method": "hist"} if device() == "cuda" else {"n_jobs": n_jobs()}


def catboost_params():
    return {"task_type": "GPU"} if device() == "cuda" else {"thread_count": n_jobs()}


def pool_map(func, items, initializer=None, initargs=(), chunksize=50):
    """map parallelo portabile; con un solo job gira nel processo corrente."""
    if n_jobs() == 1:
        if initializer:
            initializer(*initargs)
        return [func(x) for x in items]
    with mp_context().Pool(n_jobs(), initializer=initializer, initargs=initargs) as pool:
        return pool.map(func, items, chunksize=chunksize)


# Fold allenati in parallelo per famiglia di modelli; i thread di ciascuno sono n_jobs // worker.
# Misurato sul M5 (4 core P + 6 core E) con 10k righe: i boosting scalano male oltre ~3 thread
# (lgbm/xgb 1.5x più veloci con 3 worker x 3 thread che con 1 x 10, catboost 1.1x con 2 x 5);
# ExtraTrees/RF di sklearn scalano bene sui thread e restano 1 worker x tutti i thread.
FOLD_WORKERS = {"lgbm": 3, "xgb": 3, "catboost": 2}


def fold_plan(model, single_thread=False):
    """(worker, thread per worker) per allenare i fold di `model`. single_thread: modelli a un
    thread (logreg, svm, ...), un fold per core."""
    total = n_jobs()
    if single_thread:
        return total, 1
    family = model.split("_")[0]
    if device() == "cuda" and family in ("xgb", "catboost"):
        return 1, total  # una sola GPU: i fold restano sequenziali
    workers = min(FOLD_WORKERS.get(family, 1), total)
    return workers, max(1, total // workers)


def _with_threads(threads, fn, args):
    os.environ["POKEMON_N_JOBS"] = str(threads)  # letto dalle factory dei modelli nel worker
    return fn(*args)


def fold_map(fn, arg_list, workers, threads):
    """[fn(*a) for a in arg_list], con `workers` processi (joblib/loky, portabile) da `threads` thread."""
    if workers <= 1:
        return [fn(*a) for a in arg_list]
    from joblib import Parallel, delayed
    return Parallel(n_jobs=min(workers, len(arg_list)))(delayed(_with_threads)(threads, fn, a) for a in arg_list)


def describe():
    return f"platform={sys.platform} n_jobs={n_jobs()} device={device()} mp={mp_context().get_start_method()}"

**`src/features.py`**: Battle-state feature engineering, Gen 2 damage engine, label-free context (pokédex, Hidden Power types, co-occurrence, species moves).

In [ ]:
%%writefile src/features.py
"""Feature engineering per la predizione delle battaglie Pokemon Gen 2 OU.

Ogni battaglia contiene il team completo di p1, il lead di p2 e i primi 30 turni.
Le feature descrivono lo stato della battaglia al turno 30 (e in alcuni checkpoint
intermedi), la composizione dei team e lo "stile" di gioco osservato.
"""
import json
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

from compute import pool_map

STATS = ["hp", "atk", "def", "spa", "spd", "spe"]
BAD_STATUS = ["slp", "par", "tox", "psn", "frz", "brn"]
EFFECTS = ["trapped", "confusion", "substitute", "leechseed", "perish", "protect", "attract", "encore"]
BOOSTS = ["atk", "def", "spa", "spd", "spe"]
CHECKPOINTS = [5, 10, 15, 20, 25]
HAZARD_SETUP = {"spikes"}
HEAL_MOVES = {"rest", "recover", "softboiled", "milkdrink", "moonlight", "morningsun", "synthesis"}
SETUP_MOVES = {"curse", "swordsdance", "growth", "agility", "amnesia", "bellydrum", "meditate", "sharpen",
               "doubleteam", "minimize", "barrier", "acidarmor", "harden", "withdraw", "defensecurl"}
PHAZE_MOVES = {"roar", "whirlwind"}
SLEEP_MOVES = {"lovelykiss", "sleeppowder", "hypnosis", "sing", "spore"}


def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]


def get_actual_move(prev_t, t, side, last_move):
    """Mossa realmente eseguita da `side` nel turno t, oppure None.

    Il campo *_move_details riporta l'ultima mossa usata da quel pokemon anche nei turni
    in cui non agisce (circa il 25% delle mosse nel dataset). Va scartato quando:
    - il pokemon è appena entrato (switch, phazing, rimpiazzo dopo un KO);
    - il turno è di rimpiazzo dopo un KO e la mossa ripete quella del turno prima;
    - il pokemon era e resta addormentato (tranne sleeptalk/snore, rest con recupero
      di HP, o mossa diversa dall'ultima, cioè chiamata da sleeptalk);
    - il pokemon era e resta congelato.
    `last_move` è l'ultima mossa reale nota di quel pokemon.
    """
    m = t[side + "_move_details"]
    if m is None or prev_t is None:
        return m
    other = "p2" if side == "p1" else "p1"
    ps, pp = t[side + "_pokemon_state"], prev_t[side + "_pokemon_state"]
    if ps["name"] != pp["name"]:
        return None
    po, co = prev_t[other + "_pokemon_state"], t[other + "_pokemon_state"]
    pm = prev_t[side + "_move_details"]
    if po["hp_pct"] == 0 and co["name"] != po["name"] and pm is not None and pm["name"] == m["name"]:
        return None
    if pp["status"] == "slp" and ps["status"] == "slp":
        if m["name"] in ("sleeptalk", "snore"):
            return m
        if m["name"] == "rest" and ps["hp_pct"] > pp["hp_pct"]:
            return m
        if m["name"] == last_move:
            return None
    if pp["status"] == "frz" and ps["status"] == "frz":
        return None
    return m


def clean_battles(battles):
    """Copia delle battaglie con le mosse persistenti (non eseguite) messe a None."""
    out = []
    for b in battles:
        tl = []
        last = {"p1": {}, "p2": {}}
        prev = None
        for t in b["battle_timeline"]:
            t2 = dict(t)
            for side in ("p1", "p2"):
                name = t[side + "_pokemon_state"]["name"]
                m = get_actual_move(prev, t, side, last[side].get(name))
                t2[side + "_move_details"] = m
                if m is not None:
                    last[side][name] = m["name"]
            tl.append(t2)
            prev = t  # confronto con il turno grezzo (serve la mossa riportata, non quella pulita)
        out.append({**b, "battle_timeline": tl})
    return out


def load_type_chart(path):
    df = pd.read_csv(path)
    chart = {}  # dict semplice (picklabile per il multiprocessing spawn); assente = 1.0
    for a, d, m in df.itertuples(index=False):
        chart[(a, d)] = m
    return chart


def build_pokedex(battles):
    """Base stats e tipi per ogni specie, ricavati dai team p1 e dai lead p2."""
    dex = {}
    for b in battles:
        for p in b["p1_team_details"] + [b["p2_lead_details"]]:
            if p["name"] not in dex:
                dex[p["name"]] = {"types": [t for t in p["types"] if t != "notype"],
                                  **{s: p["base_" + s] for s in STATS}}
    return dex


def type_mult(chart, atk_type, def_types):
    m = 1.0
    for t in def_types:
        m *= chart.get((atk_type, t), 1.0)
    return m


class SideState:
    """Stato noto di un lato del campo: hp e status per ogni pokemon visto."""

    def __init__(self):
        self.hp = {}
        self.status = {}
        self.order = []

    def update(self, s):
        name = s["name"]
        if name not in self.hp:
            self.order.append(name)
        self.hp[name] = s["hp_pct"]
        self.status[name] = s["status"]

    def summary(self, team_size=6):
        hps = list(self.hp.values())
        n_seen = len(hps)
        fainted = sum(1 for n in self.hp if self.status[n] == "fnt" or self.hp[n] == 0)
        unseen = max(team_size - n_seen, 0)
        out = {
            "n_seen": n_seen,
            "fainted": fainted,
            "alive": team_size - fainted,
            "hp_sum": sum(hps) + unseen,
            "hp_seen_sum": sum(hps),
            "hp_alive_mean": np.mean([h for h in hps if h > 0]) if any(h > 0 for h in hps) else 0.0,
            "n_low_hp": sum(1 for h in hps if 0 < h < 0.3),
        }
        st = Counter(self.status[n] for n in self.hp if self.hp[n] > 0)
        for s in BAD_STATUS:
            out["n_" + s] = st.get(s, 0)
        out["n_statused"] = sum(st.get(s, 0) for s in BAD_STATUS)
        # pokemon "utili": vivi e non addormentati/congelati
        out["n_healthy"] = out["alive"] - st.get("slp", 0) - st.get("frz", 0)
        out["eff_hp"] = sum(h * (0.5 if self.status[n] in ("slp", "frz") else 0.8 if self.status[n] in ("par", "tox") else 1.0)
                            for n, h in self.hp.items()) + unseen
        return out


def battle_features(b, dex, chart):
    f = {"battle_id": b["battle_id"]}
    tl = b["battle_timeline"]
    p1_team = b["p1_team_details"]
    p1_names = [p["name"] for p in p1_team]

    # ---- composizione team p1 (statica) ----
    for s in STATS:
        vals = [p["base_" + s] for p in p1_team]
        f[f"p1_team_{s}_mean"] = np.mean(vals)
        f[f"p1_team_{s}_max"] = np.max(vals)
    f["p1_team_bst_mean"] = np.mean([sum(p["base_" + s] for s in STATS) for p in p1_team])
    f["p1_team_level_mean"] = np.mean([p["level"] for p in p1_team])
    f["p1_team_level_min"] = np.min([p["level"] for p in p1_team])
    f["p1_n_leftovers"] = sum(p["item"] == "leftovers" for p in p1_team)
    f["p1_n_unique_types"] = len({t for p in p1_team for t in p["types"] if t != "notype"})

    lead2 = b["p2_lead_details"]
    for s in STATS:
        f[f"p2_lead_{s}"] = lead2["base_" + s]
    f["lead_spe_diff"] = p1_team[0]["base_spe"] - lead2["base_spe"]

    # ---- simulazione dello stato lungo la timeline ----
    s1, s2 = SideState(), SideState()
    moves = {"p1": Counter(), "p2": Counter()}
    cats = {"p1": Counter(), "p2": Counter()}
    switches = {"p1": 0, "p2": 0}
    no_move = {"p1": 0, "p2": 0}
    dmg_dealt = {"p1": 0.0, "p2": 0.0}
    se_moves = {"p1": 0, "p2": 0}
    kos_turn = {"p1": [], "p2": []}
    hp_diff_traj, faint_diff_traj = [], []
    prev = {"p1": None, "p2": None}
    for t in tl:
        for side, st in (("p1", s1), ("p2", s2)):
            ps = t[side + "_pokemon_state"]
            name = ps["name"]
            if prev[side] is not None and prev[side] != name:
                switches[side] += 1
            prev[side] = name
            # danno subito da questo lato -> danno inflitto dall'avversario
            other = "p2" if side == "p1" else "p1"
            if name in st.hp:
                delta = st.hp[name] - ps["hp_pct"]
                if delta > 0:
                    dmg_dealt[other] += delta
            was_alive = st.hp.get(name, 1.0) > 0
            st.update(ps)
            if was_alive and ps["hp_pct"] == 0:
                kos_turn[other].append(t["turn"])
            m = t[side + "_move_details"]
            if m is None:
                no_move[side] += 1
            else:
                moves[side][m["name"]] += 1
                cats[side][m["category"]] += 1
                opp = t[other + "_pokemon_state"]["name"]
                if m["category"] != "status" and opp in dex:
                    if type_mult(chart, m["type"], dex[opp]["types"]) > 1:
                        se_moves[side] += 1
        a, c = s1.summary(), s2.summary()
        hp_diff_traj.append(a["hp_sum"] - c["hp_sum"])
        faint_diff_traj.append(a["fainted"] - c["fainted"])
    # checkpoint relativi all'ultimo turno (al turno 30: t25, t20, ... t5)
    for k in CHECKPOINTS:
        i = max(-1 - k, -len(hp_diff_traj))
        f[f"hp_diff_t{30 - k}"] = hp_diff_traj[i]
        f[f"faint_diff_t{30 - k}"] = faint_diff_traj[i]

    a, c = s1.summary(), s2.summary()
    for k, v in a.items():
        f["p1_" + k] = v
    for k, v in c.items():
        f["p2_" + k] = v
        f["diff_" + k] = a[k] - v
    f["hp_diff_slope"] = np.polyfit(np.arange(len(hp_diff_traj)), hp_diff_traj, 1)[0]
    f["hp_diff_last10_change"] = hp_diff_traj[-1] - hp_diff_traj[-11]
    f["hp_diff_min"] = min(hp_diff_traj)
    f["hp_diff_max"] = max(hp_diff_traj)
    f["hp_diff_mean"] = np.mean(hp_diff_traj)

    # p1 ha HP/status anche per i pokemon non ancora entrati (pieni)
    p1_unseen = [n for n in p1_names if n not in s1.hp]
    f["p1_n_unseen"] = len(p1_unseen)
    # stats dei pokemon vivi
    for side, st, extra in (("p1", s1, p1_unseen), ("p2", s2, [])):
        alive = [n for n, h in st.hp.items() if h > 0] + extra
        alive = [n for n in alive if n in dex]
        for s in ["atk", "spa", "spe", "def", "spd", "hp"]:
            f[f"{side}_alive_{s}_sum"] = sum(dex[n][s] for n in alive)
        f[f"{side}_alive_hpw_bst"] = sum(st.hp.get(n, 1.0) * sum(dex[n][s] for s in STATS) for n in alive)
    f["diff_alive_hpw_bst"] = f["p1_alive_hpw_bst"] - f["p2_alive_hpw_bst"]
    for s in ["atk", "spa", "spe"]:
        f[f"diff_alive_{s}_sum"] = f[f"p1_alive_{s}_sum"] - f[f"p2_alive_{s}_sum"]

    # ---- stato attivo al turno 30 ----
    last = tl[-1]
    act = {}
    for side in ("p1", "p2"):
        ps = last[side + "_pokemon_state"]
        act[side] = ps["name"]
        f[f"{side}_active_hp"] = ps["hp_pct"]
        f[f"{side}_active_status"] = ps["status"]
        for bst in BOOSTS:
            f[f"{side}_active_boost_{bst}"] = ps["boosts"][bst]
        f[f"{side}_active_boost_sum"] = sum(ps["boosts"].values())
        for e in EFFECTS:
            f[f"{side}_active_eff_{e}"] = int(e in ps["effects"])
        d = dex.get(ps["name"])
        for s in STATS:
            f[f"{side}_active_{s}"] = d[s] if d else np.nan
    for bst in BOOSTS:
        f[f"diff_active_boost_{bst}"] = f[f"p1_active_boost_{bst}"] - f[f"p2_active_boost_{bst}"]
    f["active_spe_diff"] = f["p1_active_spe"] - f["p2_active_spe"]
    f["p1_active_status"] = f["p1_active_status"]
    # efficacia di tipo tra gli attivi (STAB)
    d1, d2 = dex.get(act["p1"]), dex.get(act["p2"])
    if d1 and d2:
        f["p1_vs_p2_type"] = max(type_mult(chart, t, d2["types"]) for t in d1["types"])
        f["p2_vs_p1_type"] = max(type_mult(chart, t, d1["types"]) for t in d2["types"])
    else:
        f["p1_vs_p2_type"] = f["p2_vs_p1_type"] = np.nan

    # ---- matchup team: tipi STAB di p1 vivi contro p2 vivi (e viceversa) ----
    p1_alive = [n for n in p1_names if s1.hp.get(n, 1.0) > 0 and n in dex]
    p2_alive = [n for n, h in s2.hp.items() if h > 0 and n in dex]
    def best_cover(att, dfn):
        if not att or not dfn:
            return 0.0
        return np.mean([max(type_mult(chart, t, dex[d]["types"]) for a in att for t in dex[a]["types"]) for d in dfn])
    f["p1_cover_p2"] = best_cover(p1_alive, p2_alive)
    f["p2_cover_p1"] = best_cover(p2_alive, p1_alive)

    # ---- stile di gioco / mosse ----
    for side in ("p1", "p2"):
        mv = moves[side]
        f[f"{side}_switches"] = switches[side]
        f[f"{side}_no_move"] = no_move[side]
        f[f"{side}_dmg_dealt"] = dmg_dealt[side]
        f[f"{side}_se_moves"] = se_moves[side]
        f[f"{side}_n_kos"] = len(kos_turn[side])
        f[f"{side}_first_ko_turn"] = kos_turn[side][0] if kos_turn[side] else 31
        f[f"{side}_last_ko_turn"] = kos_turn[side][-1] if kos_turn[side] else 0
        for cat in ("physical", "special", "status"):
            f[f"{side}_cat_{cat}"] = cats[side][cat]
        f[f"{side}_spikes"] = int(mv["spikes"] > 0)
        f[f"{side}_rapidspin"] = mv["rapidspin"]
        f[f"{side}_heal"] = sum(mv[m] for m in HEAL_MOVES)
        f[f"{side}_setup"] = sum(mv[m] for m in SETUP_MOVES)
        f[f"{side}_phaze"] = sum(mv[m] for m in PHAZE_MOVES)
        f[f"{side}_sleepmove"] = sum(mv[m] for m in SLEEP_MOVES)
        f[f"{side}_explosion"] = mv["explosion"] + mv["selfdestruct"]
        f[f"{side}_n_unique_moves"] = len(mv)
    f["diff_dmg_dealt"] = f["p1_dmg_dealt"] - f["p2_dmg_dealt"]
    f["diff_switches"] = f["p1_switches"] - f["p2_switches"]
    # spikes attive: piazzate e non rimosse dopo l'ultimo rapid spin (approssimazione)
    f["spikes_on_p2"] = int(moves["p1"]["spikes"] > 0 and moves["p2"]["rapidspin"] == 0)
    f["spikes_on_p1"] = int(moves["p2"]["spikes"] > 0 and moves["p1"]["rapidspin"] == 0)
    f["diff_spikes"] = f["spikes_on_p2"] - f["spikes_on_p1"]

    # bag-of-pokemon (testo, vettorizzato dopo)
    f["_p1_team"] = " ".join(p1_names)
    f["_p1_alive"] = " ".join(p1_alive)
    f["_p2_seen"] = " ".join(s2.hp.keys())
    f["_p2_alive"] = " ".join(p2_alive)
    f["_p1_moves"] = " ".join(m for m, c in moves["p1"].items() for _ in range(c))
    f["_p2_moves"] = " ".join(m for m, c in moves["p2"].items() for _ in range(c))
    return f


SPECIAL_TYPES = {"fire", "water", "electric", "grass", "ice", "psychic", "dragon", "dark"}
FIXED_POWER = {"hiddenpower": 70, "return": 102, "frustration": 102, "present": 40, "magnitude": 70, "reversal": 60}
FIXED_DAMAGE = {"seismictoss", "nightshade"}  # danno = livello (100 HP)


def calc_stat(base, level, hp=False):
    """Stat Gen 2 con DV e stat exp massimi."""
    core = ((base + 15) * 2 + 63) * level // 100
    return core + level + 10 if hp else core + 5


HP_TYPES = ["fighting", "flying", "poison", "ground", "rock", "bug", "ghost", "steel",
            "fire", "water", "grass", "electric", "psychic", "ice", "dragon", "dark"]


def infer_hp_types(battles, dex, chart, min_obs=5):
    """Tipo di hiddenpower per specie, stimato dai danni osservati.

    Per ogni uso di hiddenpower in cui il difensore resta in campo, il rapporto tra
    danno osservato e danno neutro previsto stima il moltiplicatore di tipo; si sceglie
    il tipo che spiega meglio tutte le osservazioni della specie.
    """
    obs = defaultdict(list)  # specie -> [(danno osservato, difensore)]
    hp_move = {"name": "hiddenpower", "type": "normal", "category": "special", "base_power": 70, "accuracy": 1.0}
    for b in battles:
        tl = b["battle_timeline"]
        for prev, t in zip(tl, tl[1:]):
            for side, other in (("p1", "p2"), ("p2", "p1")):
                m = t[side + "_move_details"]
                if not m or m["name"] != "hiddenpower":
                    continue
                att, dfn = t[side + "_pokemon_state"]["name"], t[other + "_pokemon_state"]["name"]
                if (dfn != prev[other + "_pokemon_state"]["name"] or att != prev[side + "_pokemon_state"]["name"]
                        or att not in dex or dfn not in dex):
                    continue
                delta = prev[other + "_pokemon_state"]["hp_pct"] - t[other + "_pokemon_state"]["hp_pct"]
                if not 0.03 < delta < 1:
                    continue
                obs[att].append((delta, dfn))
    out = {}
    for sp, o in obs.items():
        if len(o) < min_obs:
            continue
        def loss(ty):
            return sum((np.log(delta) - np.log(max(damage_frac(dex, chart, sp, dfn, hp_move, hp_type=ty) / 0.925, 0.01))) ** 2
                       for delta, dfn in o)
        out[sp] = min(HP_TYPES, key=loss)
    return out


def build_cooccurrence(battles):
    """P(b nel team | a nel team) stimata dai team completi di p1."""
    pair, single = Counter(), Counter()
    for b in battles:
        names = sorted({p["name"] for p in b["p1_team_details"]})  # ordine deterministico
        for a in names:
            single[a] += 1
            for c in names:
                if a != c:
                    pair[(a, c)] += 1
    cond = defaultdict(dict)
    for (a, c), n in pair.items():
        cond[a][c] = n / single[a]
    return cond, single


def damage_frac(dex, chart, att, dfn, move, att_level=100, dfn_level=100, item=None, hp_type=None):
    """Danno atteso di `move` di att su dfn, come frazione degli HP massimi di dfn."""
    a, d = dex[att], dex[dfn]
    max_hp = calc_stat(d["hp"], dfn_level, hp=True)
    name, mtype = move["name"], move["type"]
    hp_unknown = name == "hiddenpower" and hp_type is None
    if name == "hiddenpower" and hp_type:
        mtype = hp_type
    if name in FIXED_DAMAGE:
        return att_level / max_hp * (0.0 if type_mult(chart, mtype, d["types"]) == 0 else 1.0)
    bp = move["base_power"] or FIXED_POWER.get(name, 0)
    if bp <= 1 or move["category"] == "status":
        return 0.0
    special = mtype in SPECIAL_TYPES
    A = calc_stat(a["spa" if special else "atk"], att_level)
    D = calc_stat(d["spd" if special else "def"], dfn_level)
    if item == "thickclub" and not special:
        A *= 2
    if item == "lightball" and special:
        A *= 2
    if name in ("explosion", "selfdestruct"):
        D = D / 2
    dmg = ((2 * att_level / 5 + 2) * bp * A / D / 50 + 2)
    stab = 1.5 if mtype in a["types"] else 1.0
    mult = 1.0 if hp_unknown else type_mult(chart, mtype, d["types"])
    return dmg * stab * mult * move["accuracy"] * 0.925 / max_hp  # 0.925 = media del random roll


def stab_moves(dex, name):
    """Mosse STAB generiche per pokemon di cui non conosciamo il set."""
    d = dex[name]
    return [{"name": "stab", "type": t, "category": "special" if t in SPECIAL_TYPES else "physical",
             "base_power": 80, "accuracy": 1.0} for t in d["types"]]


def build_species_moves(battles):
    """Statistiche delle mosse (reali) per specie, su tutte le battaglie.

    Restituisce (top_damaging, freq): le 2 mosse offensive più usate dalla specie, usate
    come fallback quando un pokemon non ha rivelato mosse d'attacco, e la frazione di
    battaglie in cui la specie ha usato ciascuna mossa.
    """
    used = defaultdict(Counter)
    appear = Counter()
    details = {}
    for b in battles:
        seen = defaultdict(set)
        for t in b["battle_timeline"]:
            for side in ("p1", "p2"):
                name = t[side + "_pokemon_state"]["name"]
                seen[name]  # registra la presenza
                m = t[side + "_move_details"]
                if m:
                    seen[name].add(m["name"])
                    details[m["name"]] = m
        for name, mv in seen.items():
            appear[name] += 1
            used[name].update(mv)
    freq = {sp: {m: c / appear[sp] for m, c in cnt.items()} for sp, cnt in used.items()}
    top = {}
    for sp, cnt in used.items():
        # ordinamento deterministico (conteggio, poi nome): most_common rompe i pareggi
        # nell'ordine di inserimento, che dipende dall'hash randomizzato dei set
        dmg = [m for m, _ in sorted(cnt.items(), key=lambda kv: (-kv[1], kv[0]))
               if details[m]["category"] != "status" or m in FIXED_DAMAGE]
        top[sp] = [details[m] for m in dmg[:2]]
    return top, freq


def boost_mult(stage):
    return (2 + stage) / 2 if stage >= 0 else 2 / (2 - stage)


CLERICS = {"blissey", "miltank", "chansey"}
WINCON_SETUP = {"curse", "bellydrum", "swordsdance", "growth"}


def matchup_features(b, dex, chart, hp_types, cooc, species_moves, species_freq):
    """Matrice di danno tra pokemon vivi dei due lati usando le mosse rivelate."""
    tl = b["battle_timeline"]
    p1_items = {p["name"]: p["item"] for p in b["p1_team_details"]}
    p1_level = {p["name"]: p["level"] for p in b["p1_team_details"]}
    known = {"p1": defaultdict(dict), "p2": defaultdict(dict)}
    hp = {"p1": {n: 1.0 for n in p1_items}, "p2": {}}
    status = {"p1": {}, "p2": {}}
    for t in tl:
        for side in ("p1", "p2"):
            ps = t[side + "_pokemon_state"]
            hp[side][ps["name"]] = ps["hp_pct"]
            status[side][ps["name"]] = ps["status"]
            m = t[side + "_move_details"]
            if m:
                known[side][ps["name"]][m["name"]] = m

    def best_move(side, att, dfn):
        """(danno frazionale atteso, mossa speciale?) della mossa migliore di att su dfn."""
        mv = list(known[side][att].values())
        dmg_moves = [m for m in mv if m["category"] != "status" or m["name"] in FIXED_DAMAGE]
        if not dmg_moves:
            # nessuna mossa d'attacco rivelata: le più usate dalla specie, altrimenti STAB generico
            dmg_moves = species_moves.get(att) or stab_moves(dex, att)
        lv_a = p1_level.get(att, 100) if side == "p1" else 100
        lv_d = 100 if side == "p1" else p1_level.get(dfn, 100)
        item = p1_items.get(att) if side == "p1" else None
        best = (0.0, False)
        for m in dmg_moves:
            d = damage_frac(dex, chart, att, dfn, m, lv_a, lv_d, item, hp_types.get(att))
            if d > best[0]:
                mtype = hp_types.get(att, m["type"]) if m["name"] == "hiddenpower" else m["type"]
                best = (d, mtype in SPECIAL_TYPES)
        if status[side].get(att) == "brn" and not best[1]:
            best = (best[0] * 0.5, best[1])
        return best

    def best_dmg(side, att, dfn):
        return best_move(side, att, dfn)[0]

    # distribuzione dei pokemon di p2 non ancora visti (co-occorrenze nei team p1)
    seen2 = [n for n in hp["p2"] if n in dex]
    n_unseen = max(6 - len(hp["p2"]), 0)
    probs = Counter()
    for a in seen2:
        for c, p in cooc.get(a, {}).items():
            if c not in hp["p2"] and c in dex:
                probs[c] += p
    tot = sum(probs.values())
    probs = {c: p / tot for c, p in probs.items()} if tot > 0 else {}

    f = {}
    alive = {s: [n for n, h in hp[s].items() if h > 0 and n in dex] for s in ("p1", "p2")}
    for side, other in (("p1", "p2"), ("p2", "p1")):
        A, D = alive[side], alive[other]
        if not A or not D:
            for k in ["best_dmg_mean", "ohko_frac", "2hko_frac", "fast_ohko_frac", "checked_frac", "threat_max"]:
                f[f"{side}_{k}"] = 0.0
            continue
        M = np.array([[best_dmg(side, a, d) for d in D] for a in A])  # danno frazionale su HP max
        rem = np.array([hp[other][d] for d in D])
        spe_a = np.array([dex[a]["spe"] for a in A])[:, None]
        spe_d = np.array([dex[d]["spe"] for d in D])[None, :]
        para_a = np.array([status[side].get(a) == "par" for a in A])[:, None]
        faster = (spe_a * np.where(para_a, 0.25, 1.0)) > spe_d
        best = M.max(axis=0)
        f[f"{side}_best_dmg_mean"] = best.mean()
        f[f"{side}_ohko_frac"] = (best >= rem).mean()
        f[f"{side}_2hko_frac"] = (best * 2 >= rem).mean()
        f[f"{side}_fast_ohko_frac"] = ((M >= rem[None, :]) & faster).any(axis=0).mean()
        # quanti avversari vengono "controllati": qualcuno li colpisce per >=40%
        f[f"{side}_checked_frac"] = (best >= 0.4).mean()
        f[f"{side}_threat_max"] = M.max()
        htk = np.ceil(rem / np.maximum(best, 0.05))
        n_targets = len(D)
        if side == "p1" and n_unseen and probs:
            # slot virtuali per i pokemon di p2 non visti (pieni), pesati con le co-occorrenze
            exp_htk = sum(p * np.ceil(1 / max(max(best_dmg("p1", a, c) for a in A), 0.05)) for c, p in probs.items())
            htk = np.append(htk, [exp_htk] * n_unseen)
            n_targets += n_unseen
        f[f"{side}_hits_to_kill"] = htk.sum()
        f[f"{side}_hits_to_kill_per_mon"] = htk.sum() / n_targets
    for k in ["best_dmg_mean", "ohko_frac", "2hko_frac", "fast_ohko_frac", "checked_frac", "threat_max",
              "hits_to_kill", "hits_to_kill_per_mon"]:
        f[f"diff_{k}"] = f.get(f"p1_{k}", 0) - f.get(f"p2_{k}", 0)

    # duello attuale degli attivi al turno 30, con boost e paralisi
    last = tl[-1]
    a1, a2 = last["p1_pokemon_state"]["name"], last["p2_pokemon_state"]["name"]
    if a1 in dex and a2 in dex and hp["p1"][a1] > 0 and hp["p2"][a2] > 0:
        b1, b2 = last["p1_pokemon_state"]["boosts"], last["p2_pokemon_state"]["boosts"]
        def boosted(side, att, dfn, ba, bd):
            d, special = best_move(side, att, dfn)
            off, dfs = ("spa", "spd") if special else ("atk", "def")
            return d * boost_mult(ba[off]) / boost_mult(bd[dfs])
        d12, d21 = boosted("p1", a1, a2, b1, b2), boosted("p2", a2, a1, b2, b1)
        if "substitute" in last["p2_pokemon_state"]["effects"]:
            d12 *= 0.75  # il primo colpo va sul substitute
        if "substitute" in last["p1_pokemon_state"]["effects"]:
            d21 *= 0.75
        f["active_p1_dmg"] = d12
        f["active_p2_dmg"] = d21
        f["active_p1_turns_to_ko"] = hp["p2"][a2] / max(d12, 0.01)
        f["active_p2_turns_to_ko"] = hp["p1"][a1] / max(d21, 0.01)
        f["active_duel"] = np.log(f["active_p2_turns_to_ko"]) - np.log(f["active_p1_turns_to_ko"])
        sp1 = dex[a1]["spe"] * boost_mult(b1["spe"]) * (0.25 if status["p1"].get(a1) == "par" else 1.0)
        sp2 = dex[a2]["spe"] * boost_mult(b2["spe"]) * (0.25 if status["p2"].get(a2) == "par" else 1.0)
        f["active_p1_faster"] = float(np.sign(sp1 - sp2))
    else:
        for k in ["active_p1_dmg", "active_p2_dmg", "active_p1_turns_to_ko", "active_p2_turns_to_ko", "active_duel",
                  "active_p1_faster"]:
            f[k] = np.nan

    # ---- spikes cronologici (rapid spin fallisce contro ghost) ----
    spikes = {"p1": False, "p2": False}  # spikes sul lato di ...
    for t in tl:
        for side, other in (("p1", "p2"), ("p2", "p1")):
            m = t[side + "_move_details"]
            if not m:
                continue
            if m["name"] == "spikes":
                spikes[other] = True
            elif m["name"] == "rapidspin":
                opp = t[other + "_pokemon_state"]["name"]
                if "ghost" not in dex.get(opp, {}).get("types", []):
                    spikes[side] = False
    f["spikes_now_p1"] = int(spikes["p1"])
    f["spikes_now_p2"] = int(spikes["p2"])
    f["spikes_now_diff"] = f["spikes_now_p2"] - f["spikes_now_p1"]
    f["p1_has_spinner_alive"] = int(any("rapidspin" in known["p1"][n] for n in alive["p1"]))
    f["p2_has_spinner_alive"] = int(any("rapidspin" in known["p2"][n] for n in alive["p2"]))

    # ---- sonno: da rest (temporaneo) o indotto; turni di sonno "consumati" al turno 30 ----
    # Il contatore del sonno in Gen 2 scende solo quando il pokemon è in campo e prova ad agire.
    for side in ("p1", "p2"):
        rest_sleep, induced_sleep, induced_burned, rest_waking = 0, 0, 0, 0
        last_rest, burned = {}, {}
        berry = 0
        prev = None
        for t in tl:
            ps = t[side + "_pokemon_state"]
            name = ps["name"]
            m = t[side + "_move_details"]
            if m and m["name"] == "rest":
                if ps["status"] == "slp":
                    last_rest[name] = t["turn"]
                    burned[name] = 0
                else:
                    berry += 1  # rest seguito da risveglio immediato: mintberry / miracleberry
            elif ps["status"] == "slp":
                if prev is not None and prev["name"] == name and prev["status"] == "slp":
                    burned[name] = burned.get(name, 0) + 1
                else:
                    burned.setdefault(name, 0)
            else:
                # sveglio: dimentica rest e contatore
                last_rest.pop(name, None)
                burned.pop(name, None)
            if (prev is not None and prev["name"] == name and prev["status"] in ("par", "psn", "tox", "brn", "frz")
                    and ps["status"] == "nostatus" and ps["hp_pct"] > 0 and not (m and m["name"] == "rest")):
                berry += 1  # status curato senza rest: miracleberry / przcureberry ecc.
            prev = ps
        for n, st in status[side].items():
            if st == "slp" and hp[side].get(n, 0) > 0:
                if n in last_rest:
                    rest_sleep += 1
                    rest_waking += int(burned.get(n, 0) >= 2)
                else:
                    induced_sleep += 1
                    induced_burned += burned.get(n, 0)
        f[f"{side}_rest_sleep"] = rest_sleep
        f[f"{side}_rest_waking"] = rest_waking
        f[f"{side}_induced_sleep"] = induced_sleep
        f[f"{side}_induced_sleep_turns"] = induced_burned
        f[f"{side}_berry_used"] = berry
        # mosse note per pokemon vivi (quanto del set è stato rivelato)
        f[f"{side}_known_moves_alive"] = sum(len(known[side][n]) for n in alive[side])
    f["diff_induced_sleep"] = f["p1_induced_sleep"] - f["p2_induced_sleep"]

    # ---- win condition di fine partita: setup non phazabile, cleric ----
    def has_move(side, n, moves, min_freq=0.3):
        if any(m in known[side][n] for m in moves):
            return True
        return sum(species_freq.get(n, {}).get(m, 0) for m in moves) >= min_freq
    for side, other in (("p1", "p2"), ("p2", "p1")):
        setup = any(any(m in known[side][n] for m in WINCON_SETUP) for n in alive[side])
        phazer = any(has_move(other, n, PHAZE_MOVES) for n in alive[other])
        f[f"{side}_setup_alive"] = int(setup)
        f[f"{side}_phazer_alive"] = int(phazer)
        f[f"{side}_unphazed_setup"] = int(setup and not phazer)
        cleric = any(n in CLERICS or "healbell" in known[side][n] for n in alive[side])
        f[f"{side}_cleric_alive"] = int(cleric)
        n_st = sum(1 for n in alive[side] if status[side].get(n) in ("par", "psn", "tox", "brn", "slp", "frz"))
        f[f"{side}_status_burden"] = n_st * (0.3 if cleric else 1.0)
    f["unphazed_setup_adv"] = f["p1_unphazed_setup"] - f["p2_unphazed_setup"]
    f["diff_status_burden"] = f["p1_status_burden"] - f["p2_status_burden"]

    # ---- pokemon di p2 non ancora visti, stimati dalle co-occorrenze nei team p1 ----
    f["p2_n_unseen"] = n_unseen
    if n_unseen and probs:
        for s in STATS:
            f[f"p2_unseen_exp_{s}"] = sum(p * dex[c][s] for c, p in probs.items())
        f["p2_unseen_exp_bst"] = sum(f[f"p2_unseen_exp_{s}"] for s in STATS)
        # quanto i vivi di p1 colpiscono (STAB) e sono colpiti dai probabili non visti
        f["p1_cover_p2_unseen"] = sum(
            p * max((type_mult(chart, t, dex[c]["types"]) for a in alive["p1"] for t in dex[a]["types"]), default=1.0)
            for c, p in probs.items())
        f["p2_unseen_threat"] = sum(
            p * np.mean([max(type_mult(chart, t, dex[a]["types"]) for t in dex[c]["types"]) for a in alive["p1"]] or [1.0])
            for c, p in probs.items())
        f["p2_unseen_p_snorlax"] = probs.get("snorlax", 0.0)
    else:
        for s in STATS + ["bst"]:
            f[f"p2_unseen_exp_{s}"] = 0.0
        f["p1_cover_p2_unseen"] = f["p2_unseen_threat"] = f["p2_unseen_p_snorlax"] = 0.0
    f["p2_unseen_exp_bst_tot"] = f["p2_unseen_exp_bst"] * n_unseen
    return f


def build_context(battles, dex, chart):
    """Statistiche globali (senza etichette) calcolate sulle battaglie già pulite."""
    species_moves, species_freq = build_species_moves(battles)
    return dict(hp_types=infer_hp_types(battles, dex, chart), cooc=build_cooccurrence(battles)[0],
                species_moves=species_moves, species_freq=species_freq)


def symmetrize_unseen(f):
    """Aggiunge a p2 il valore atteso dei pokemon non visti, come già avviene per p1."""
    n = f["p2_n_unseen"]
    for s in ["atk", "spa", "spe", "def", "spd", "hp"]:
        f[f"p2_alive_{s}_sum"] += n * f[f"p2_unseen_exp_{s}"]
    f["p2_alive_hpw_bst"] += n * f["p2_unseen_exp_bst"]
    f["diff_alive_hpw_bst"] = f["p1_alive_hpw_bst"] - f["p2_alive_hpw_bst"]
    for s in ["atk", "spa", "spe"]:
        f[f"diff_alive_{s}_sum"] = f[f"p1_alive_{s}_sum"] - f[f"p2_alive_{s}_sum"]
    return f


def build_frame(battles, dex, chart, ctx=None, truncate=None):
    """Feature per battaglia. Le battaglie devono essere già passate da `clean_battles`.

    `truncate=t` calcola le feature come se la timeline finisse al turno t (augmentation).
    """
    ctx = ctx or build_context(battles, dex, chart)
    if truncate:
        battles = [{**b, "battle_timeline": b["battle_timeline"][:truncate]} for b in battles]
    # le battaglie sono indipendenti: calcolo parallelo (portabile, vedi compute.py)
    return pd.DataFrame(pool_map(_frame_row, battles, _init_frame_worker, (dex, chart, ctx), chunksize=200))


_W = {}


def _init_frame_worker(dex, chart, ctx):
    _W.update(dex=dex, chart=chart, ctx=ctx)


def _frame_row(b):
    dex, chart = _W["dex"], _W["chart"]
    return symmetrize_unseen({**battle_features(b, dex, chart), **matchup_features(b, dex, chart, **_W["ctx"])})

**`src/train.py`**: Feature cache (turn 30 + training snapshots at turns 20-28) and bag-of-words columns.

In [ ]:
%%writefile src/train.py
"""Training + CV dei modelli e generazione della submission.

Uso: python src/train.py [--tune N_TRIALS]
"""
import argparse
import os
import sys

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

sys.path.insert(0, os.path.dirname(__file__))
from compute import catboost_params, configure, describe, xgb_params
from features import build_context, build_frame, build_pokedex, clean_battles, load_jsonl, load_type_chart

ROOT = os.path.join(os.path.dirname(__file__), "..")
DATA = os.environ.get("POKEMON_DATA", os.path.join(ROOT, "data"))  # su Kaggle: /kaggle/input/...
OUT = os.environ.get("POKEMON_OUT", os.path.join(ROOT, "outputs"))
SEED = 42
N_FOLDS = 5


SNAP_TURNS = (20, 22, 24, 26, 28)
# Pulizia delle mosse persistenti (features.clean_battles): corretta come meccanica di gioco ma
# peggiora la CV di ~0.4 punti (le mosse "ripetute" portano segnale), quindi disattivata.
CLEAN_MOVES = False


def _load_clean():
    prep = clean_battles if CLEAN_MOVES else (lambda x: x)
    train = prep(load_jsonl(os.path.join(DATA, "train.jsonl")))
    test = prep(load_jsonl(os.path.join(DATA, "test.jsonl")))
    chart = load_type_chart(os.path.join(DATA, "type_chart_gen2.csv"))
    dex = build_pokedex(train + test)
    return train, test, dex, chart, build_context(train + test, dex, chart)


def get_features(snapshots=False):
    """Feature al turno 30 (train+test). Con snapshots=True anche le feature dei soli
    battle di train troncati ai turni SNAP_TURNS, per l'augmentation (colonna snap_turn)."""
    os.makedirs(OUT, exist_ok=True)  # outputs/ non è versionata: su un clone pulito non esiste
    cache = os.path.join(OUT, "features.pkl")
    snap_cache = os.path.join(OUT, "features_snap.pkl")
    if os.path.exists(cache) and (not snapshots or os.path.exists(snap_cache)):
        df = pd.read_pickle(cache)
        return (df, pd.read_pickle(snap_cache)) if snapshots else df
    train, test, dex, chart, ctx = _load_clean()
    df = build_frame(train + test, dex, chart, ctx)
    df["is_test"] = [0] * len(train) + [1] * len(test)
    df["player_won"] = [b["player_won"] for b in train] + [-1] * len(test)
    df.to_pickle(cache)
    if not snapshots:
        return df
    snaps = []
    for t in SNAP_TURNS:
        s = build_frame(train, dex, chart, ctx, truncate=t)
        s["is_test"], s["player_won"], s["snap_turn"] = 0, [b["player_won"] for b in train], t
        snaps.append(s)
    snap = pd.concat(snaps, ignore_index=True)
    snap.to_pickle(snap_cache)
    return df, snap


def add_bags(df, min_df=20):
    """Bag-of-pokemon / bag-of-moves come colonne di conteggio."""
    parts = [df]
    for col in [c for c in df.columns if c.startswith("_")]:
        cv = CountVectorizer(token_pattern=r"\S+", min_df=min_df)
        m = cv.fit_transform(df[col]).toarray()
        parts.append(pd.DataFrame(m, columns=[f"{col[1:]}__{v}" for v in cv.get_feature_names_out()], index=df.index))
    out = pd.concat(parts, axis=1).drop(columns=[c for c in df.columns if c.startswith("_")])
    for c in ["p1_active_status", "p2_active_status"]:
        out = pd.concat([out.drop(columns=c), pd.get_dummies(out[c], prefix=c, dtype=int)], axis=1)
    return out


def cv_model(name, make, X, y, Xte):
    skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)
    oof = np.zeros(len(y))
    pte = np.zeros(len(Xte))
    for tr, va in skf.split(X, y):
        m = make()
        m.fit(X.iloc[tr], y[tr])
        oof[va] = m.predict_proba(X.iloc[va])[:, 1]
        pte += m.predict_proba(Xte)[:, 1] / N_FOLDS
    acc = ((oof > 0.5) == y).mean()
    print(f"{name:10s} CV acc = {acc:.4f}", flush=True)
    return oof, pte, acc


LGB_PARAMS = dict(n_estimators=600, learning_rate=0.02, num_leaves=15, min_child_samples=30,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.3, reg_lambda=5.0,
                  random_state=SEED, verbose=-1)


def tune_lgb(X, y, n_trials):
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)

    def objective(trial):
        p = dict(n_estimators=trial.suggest_int("n_estimators", 200, 1500),
                 learning_rate=trial.suggest_float("learning_rate", 0.005, 0.08, log=True),
                 num_leaves=trial.suggest_int("num_leaves", 4, 64),
                 min_child_samples=trial.suggest_int("min_child_samples", 5, 100),
                 subsample=trial.suggest_float("subsample", 0.5, 1.0), subsample_freq=1,
                 colsample_bytree=trial.suggest_float("colsample_bytree", 0.05, 0.8),
                 reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 30, log=True),
                 reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
                 random_state=SEED, verbose=-1)
        accs = []
        for tr, va in skf.split(X, y):
            m = lgb.LGBMClassifier(**p).fit(X.iloc[tr], y[tr])
            accs.append(((m.predict_proba(X.iloc[va])[:, 1] > 0.5) == y[va]).mean())
        return np.mean(accs)

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials, callbacks=[
        lambda s, t: print(f"trial {t.number}: {t.value:.4f} (best {s.best_value:.4f})", flush=True)])
    print("best tuned lgb", study.best_value, study.best_params)
    return {**study.best_params, "subsample_freq": 1, "random_state": SEED, "verbose": -1}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--tune", type=int, default=0)
    ap.add_argument("--n-jobs", type=int, default=0, help="processi/thread (default: tutti i core)")
    ap.add_argument("--device", default=None, choices=["auto", "cpu", "cuda"])
    args = ap.parse_args()
    configure(args.n_jobs, args.device)
    print(describe())
    os.makedirs(OUT, exist_ok=True)

    df = add_bags(get_features())
    tr, te = df[df.is_test == 0], df[df.is_test == 1]
    y = tr.player_won.values
    drop = ["battle_id", "is_test", "player_won"]
    X, Xte = tr.drop(columns=drop), te.drop(columns=drop)
    X.columns = Xte.columns = [c.replace(" ", "_") for c in X.columns]
    print("shape", X.shape, flush=True)

    lgb_params = tune_lgb(X, y, args.tune) if args.tune else LGB_PARAMS
    models = {
        "logreg": lambda: make_pipeline(StandardScaler(), LogisticRegression(C=0.02, max_iter=3000)),
        "lgbm": lambda: lgb.LGBMClassifier(**lgb_params),
        "xgb": lambda: xgb.XGBClassifier(n_estimators=700, learning_rate=0.02, max_depth=4, subsample=0.8,
                                         colsample_bytree=0.3, min_child_weight=5, reg_lambda=5.0,
                                         random_state=SEED, **xgb_params()),
        "catboost": lambda: CatBoostClassifier(iterations=1500, learning_rate=0.03, depth=6, l2_leaf_reg=5,
                                               random_seed=SEED, verbose=0, **catboost_params()),
    }
    Xf, Xtef = X.fillna(0), Xte.fillna(0)
    res = {}
    for name, make in models.items():
        a, b = (Xf, Xtef) if name == "logreg" else (X, Xte)
        res[name] = cv_model(name, make, a, y, b)

    # ensemble: media delle probabilità, e media dei migliori
    names = sorted(res, key=lambda n: -res[n][2])
    best_combo, best_acc = None, 0
    for k in range(1, len(names) + 1):
        oof = np.mean([res[n][0] for n in names[:k]], axis=0)
        acc = ((oof > 0.5) == y).mean()
        print(f"ensemble top{k} {names[:k]} CV acc = {acc:.4f}")
        if acc > best_acc:
            best_combo, best_acc = names[:k], acc
    pte = np.mean([res[n][1] for n in best_combo], axis=0)
    sub = pd.DataFrame({"battle_id": te.battle_id.values, "player_won": (pte > 0.5).astype(int)})
    sub.to_csv(os.path.join(OUT, "submission.csv"), index=False)
    print(f"submission salvata ({best_combo}, CV {best_acc:.4f}) -> outputs/submission.csv")

    # importanza feature (lgbm full fit)
    m = lgb.LGBMClassifier(**lgb_params).fit(X, y)
    imp = pd.Series(m.booster_.feature_importance("gain"), index=X.columns).sort_values(ascending=False)
    imp.to_csv(os.path.join(OUT, "feature_importance.csv"))
    print(imp.head(25))


if __name__ == "__main__":
    main()

**`src/simulate.py`**: Monte Carlo simulator: plays every battle to the end 100 times from the last observed state.

In [ ]:
%%writefile src/simulate.py
"""Pseudo Monte Carlo: simula il resto della battaglia a partire dal turno 30.

Riusa il motore di danno di features.py (stats Gen 2, mosse rivelate, tipo di hiddenpower
inferito) e campiona le incertezze: pokemon di p2 non visti (dalle co-occorrenze),
random roll, crit, miss, durata del sonno, paralisi, scongelamento. Ogni battaglia viene
giocata fino alla fine N volte; le statistiche delle simulazioni diventano feature per lo stacking.

Uso: python src/simulate.py [--n-sims 40]  ->  outputs/mc_features.pkl
"""
import argparse
import os
import random
import sys
import time

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(__file__))
from compute import configure, describe, pool_map
from features import (HEAL_MOVES, FIXED_DAMAGE, build_cooccurrence, build_pokedex, clean_battles,
                      damage_frac, infer_hp_types, load_jsonl, load_type_chart, stab_moves)

ROOT = os.path.join(os.path.dirname(__file__), "..")
DATA = os.environ.get("POKEMON_DATA", os.path.join(ROOT, "data"))  # su Kaggle: /kaggle/input/...
OUT = os.environ.get("POKEMON_OUT", os.path.join(ROOT, "outputs"))
MAX_TURNS = 250
SNAP_TURNS = (24, 26, 28)
CRIT = 0.066

_G = {}  # stato globale dei worker (dex, chart, ...)


def _prep(battles):
    return clean_battles(battles) if _G.get("clean") else battles


def build_globals(clean=False):
    """Dati condivisi dalle simulazioni, calcolati una volta nel processo padre.
    clean=True: timeline con le sole mosse realmente eseguite (spikes, rest, mosse note)."""
    _G["clean"] = clean
    battles = _prep(load_jsonl(os.path.join(DATA, "train.jsonl")) + load_jsonl(os.path.join(DATA, "test.jsonl")))
    chart = load_type_chart(os.path.join(DATA, "type_chart_gen2.csv"))
    dex = build_pokedex(battles)
    return dict(clean=clean, dex=dex, chart=chart, hp_types=infer_hp_types(battles, dex, chart),
                cooc=build_cooccurrence(battles)[0])


def _set_globals(g):
    """Initializer dei worker: con spawn (Windows) i dati arrivano da qui, con fork sono già ereditati."""
    _G.update(g)


def boost_mult(stage):
    return (2 + stage) / 2 if stage >= 0 else 2 / (2 - stage)


def parse_state(b):
    """Stato noto al turno 30: per ogni pokemon hp, status, mosse note, info sul sonno."""
    tl = b["battle_timeline"]
    side = {"p1": {}, "p2": {}}
    for p in b["p1_team_details"]:
        side["p1"][p["name"]] = dict(hp=1.0, status="nostatus", moves={}, level=p["level"], item=p["item"],
                                     rest_turn=None, slp_since=None, tox_turns=0)
    spikes = {"p1": False, "p2": False}
    for t in tl:
        for s, o in (("p1", "p2"), ("p2", "p1")):
            ps = t[s + "_pokemon_state"]
            mon = side[s].setdefault(ps["name"], dict(hp=1.0, status="nostatus", moves={}, level=100,
                                                      item="leftovers", rest_turn=None, slp_since=None, tox_turns=0))
            mon["hp"], prev_status, mon["status"] = ps["hp_pct"], mon["status"], ps["status"]
            if ps["status"] == "slp" and prev_status != "slp":
                mon["slp_since"] = t["turn"]
            mon["tox_turns"] = mon["tox_turns"] + 1 if ps["status"] == "tox" else 0
            m = t[s + "_move_details"]
            if m:
                mon["moves"][m["name"]] = m
                if m["name"] == "rest":
                    mon["rest_turn"] = t["turn"]
                elif m["name"] == "spikes":
                    spikes[o] = True
                elif m["name"] == "rapidspin":
                    if "ghost" not in _G["dex"].get(t[o + "_pokemon_state"]["name"], {}).get("types", []):
                        spikes[s] = False
    last = tl[-1]
    active = {s: last[s + "_pokemon_state"]["name"] for s in ("p1", "p2")}
    boosts = {s: last[s + "_pokemon_state"]["boosts"] for s in ("p1", "p2")}
    return side, active, boosts, spikes


class Battle:
    """Una singola simulazione. I pokemon sono indicizzati 0..n-1 per lato."""

    def __init__(self, b, side, active, boosts, spikes, rng):
        dex, chart, hp_types = _G["dex"], _G["chart"], _G["hp_types"]
        self.rng = rng
        mons = {s: {n: dict(v) for n, v in side[s].items() if n in dex} for s in ("p1", "p2")}
        # completa il team di p2 campionando i non visti dalle co-occorrenze
        n_missing = 6 - len(side["p2"])
        if n_missing > 0:
            probs = {}
            for a in mons["p2"]:
                for c, p in _G["cooc"].get(a, {}).items():
                    if c not in side["p2"] and c in dex:
                        probs[c] = probs.get(c, 0) + p
            cands = sorted(probs)  # ordine indipendente dall'hash randomizzato
            w = np.array([probs[c] for c in cands])
            if len(cands):
                for c in rng_np_choice(rng, cands, w, min(n_missing, len(cands))):
                    mons["p2"][c] = dict(hp=1.0, status="nostatus", moves={}, level=100, item="leftovers",
                                         rest_turn=None, slp_since=None, tox_turns=0)
        self.names = {s: list(mons[s]) for s in ("p1", "p2")}
        self.hp = {s: [mons[s][n]["hp"] for n in self.names[s]] for s in ("p1", "p2")}
        self.status = {s: [mons[s][n]["status"] for n in self.names[s]] for s in ("p1", "p2")}
        self.tox = {s: [mons[s][n]["tox_turns"] for n in self.names[s]] for s in ("p1", "p2")}
        self.leftovers = {s: [mons[s][n]["item"] == "leftovers" for n in self.names[s]] for s in ("p1", "p2")}
        self.spe = {s: [dex[n]["spe"] for n in self.names[s]] for s in ("p1", "p2")}
        self.flying = {s: ["flying" in dex[n]["types"] for n in self.names[s]] for s in ("p1", "p2")}
        self.heal = {s: [next((m for m in mons[s][n]["moves"] if m in HEAL_MOVES), None) for n in self.names[s]]
                     for s in ("p1", "p2")}
        self.spikes = dict(spikes)
        # turni di sonno rimanenti (now = ultimo turno osservato, 30 o lo snapshot)
        now = b["battle_timeline"][-1]["turn"]
        self.sleep = {}
        for s in ("p1", "p2"):
            rem = []
            for n in self.names[s]:
                m = mons[s][n]
                if m["status"] != "slp":
                    rem.append(0)
                elif m["rest_turn"] is not None and m["rest_turn"] >= (m["slp_since"] or 0):
                    rem.append(max(1, 2 - (now - m["rest_turn"])))
                else:
                    slept = now - (m["slp_since"] or now)
                    rem.append(rng.randint(1, max(1, 7 - slept)))
            self.sleep[s] = rem
        # matrici di danno: (danno grezzo frazionale, accuratezza) della mossa migliore
        self.dmg = {}
        for s, o in (("p1", "p2"), ("p2", "p1")):
            M = []
            for a in self.names[s]:
                ma = mons[s][a]
                mv = [m for m in ma["moves"].values() if m["category"] != "status" or m["name"] in FIXED_DAMAGE]
                if not mv:
                    # STAB generico: le mosse più usate dalla specie (build_species_moves) peggiorano
                    # l'AUC di mc_hp_diff da 0.693 a 0.684
                    mv = stab_moves(dex, a)
                row = []
                for d in self.names[o]:
                    best = (0.0, 1.0)
                    for m in mv:
                        acc = m["accuracy"] or 1.0
                        raw = damage_frac(dex, chart, a, d, dict(m, accuracy=1.0), ma["level"],
                                          mons[o][d]["level"], ma["item"], hp_types.get(a)) / 0.925
                        if raw * acc > best[0] * best[1]:
                            best = (raw, acc)
                    row.append(best)
                M.append(row)
            self.dmg[s] = M
        self.active = {s: self.names[s].index(active[s]) if active[s] in self.names[s] else 0 for s in ("p1", "p2")}
        self.boost = {s: (max(boosts[s]["atk"], boosts[s]["spa"]), max(boosts[s]["def"], boosts[s]["spd"]),
                          boosts[s]["spe"]) for s in ("p1", "p2")}
        self.turns = 0

    def alive(self, s):
        return [i for i, h in enumerate(self.hp[s]) if h > 0]

    def eff_speed(self, s):
        i = self.active[s]
        sp = self.spe[s][i] * boost_mult(self.boost[s][2])
        return sp * 0.25 if self.status[s][i] == "par" else sp

    def matchup(self, s, o, i):
        """Punteggio del pokemon i di s contro l'attivo avversario."""
        j = self.active[o]
        mine = self.dmg[s][i][j][0] * self.dmg[s][i][j][1]
        theirs = self.dmg[o][j][i][0] * self.dmg[o][j][i][1]
        return mine / max(self.hp[o][j], 0.05) - theirs / max(self.hp[s][i], 0.05)

    def switch_in(self, s, i):
        self.active[s] = i
        self.boost[s] = (0, 0, 0)
        if self.status[s][i] == "tox":
            self.tox[s][i] = 0
        if self.spikes[s] and not self.flying[s][i]:
            self.hp[s][i] = max(0.0, self.hp[s][i] - 0.125)

    def choose_replacement(self, s, o):
        alive = self.alive(s)
        return max(alive, key=lambda i: self.matchup(s, o, i) + self.rng.gauss(0, 0.3))

    def can_act(self, s):
        i = self.active[s]
        st = self.status[s][i]
        if st == "slp":
            self.sleep[s][i] -= 1
            if self.sleep[s][i] <= 0:
                self.status[s][i] = "nostatus"
            return False
        if st == "frz":
            if self.rng.random() < 0.1:
                self.status[s][i] = "nostatus"
            return False
        if st == "par" and self.rng.random() < 0.25:
            return False
        return True

    def act(self, s, o):
        """Azione del lato s: switch volontario, cura o attacco."""
        i, j = self.active[s], self.active[o]
        # switch volontario se il matchup è pessimo ed esiste un'alternativa nettamente migliore
        if self.matchup(s, o, i) < -2.0:
            alts = [k for k in self.alive(s) if k != i]
            if alts:
                k = max(alts, key=lambda k: self.matchup(s, o, k))
                if self.matchup(s, o, k) > self.matchup(s, o, i) + 1.5 and self.rng.random() < 0.5:
                    self.switch_in(s, k)
                    return
        if not self.can_act(s):
            return
        heal = self.heal[s][i]
        if heal and self.hp[s][i] < 0.4:
            if heal == "rest":
                self.hp[s][i], self.status[s][i], self.sleep[s][i] = 1.0, "slp", 2
            else:
                self.hp[s][i] = min(1.0, self.hp[s][i] + 0.5)
            return
        raw, acc = self.dmg[s][i][j]
        if raw <= 0 or self.rng.random() > acc:
            return
        dmg = raw * self.rng.uniform(0.85, 1.0) * boost_mult(self.boost[s][0]) / boost_mult(self.boost[o][1])
        if self.rng.random() < CRIT:
            dmg *= 2
        self.hp[o][j] = max(0.0, self.hp[o][j] - dmg)

    def residual(self, s):
        i = self.active[s]
        if self.hp[s][i] <= 0:
            return
        st = self.status[s][i]
        if st in ("psn", "brn"):
            self.hp[s][i] -= 0.125
        elif st == "tox":
            self.tox[s][i] += 1
            self.hp[s][i] -= self.tox[s][i] / 16
        if self.leftovers[s][i]:
            self.hp[s][i] += 1 / 16
        self.hp[s][i] = min(1.0, max(0.0, self.hp[s][i]))

    def run(self):
        while self.turns < MAX_TURNS:
            self.turns += 1
            for s, o in (("p1", "p2"), ("p2", "p1")):
                if self.hp[s][self.active[s]] <= 0:
                    if not self.alive(s):
                        return self.result()
                    self.switch_in(s, self.choose_replacement(s, o))
            s1, s2 = self.eff_speed("p1"), self.eff_speed("p2")
            first = "p1" if s1 > s2 or (s1 == s2 and self.rng.random() < 0.5) else "p2"
            second = "p2" if first == "p1" else "p1"
            self.act(first, second)
            if self.hp[second][self.active[second]] > 0:
                self.act(second, first)
            self.residual(first)
            self.residual(second)
            if not self.alive("p1") or not self.alive("p2"):
                return self.result()
        return self.result()

    def result(self):
        a1, a2 = len(self.alive("p1")), len(self.alive("p2"))
        h1, h2 = sum(self.hp["p1"]), sum(self.hp["p2"])
        win = 1.0 if a2 == 0 and a1 > 0 else 0.0 if a1 == 0 else float(h1 > h2)
        return win, a1, a2, h1, h2, self.turns, float(self.turns >= MAX_TURNS)


def rng_np_choice(rng, items, w, k):
    """Campionamento senza reinserimento pesato, usando random.Random (riproducibile)."""
    items, w = list(items), list(w)
    out = []
    for _ in range(k):
        r, acc = rng.random() * sum(w), 0.0
        for idx, wi in enumerate(w):
            acc += wi
            if acc >= r:
                break
        out.append(items.pop(idx))
        w.pop(idx)
    return out


def simulate_battle(args):
    b, n_sims = args
    side, active, boosts, spikes = parse_state(_prep([b])[0])
    rng = random.Random(b["battle_id"] * 7919 + int(b.get("player_won", -1) == -1) + 100 * len(b["battle_timeline"]))
    res = np.array([Battle(b, side, active, boosts, spikes, rng).run() for _ in range(n_sims)])
    win, a1, a2, h1, h2, turns, capped = res.T
    return {"mc_win": win.mean(), "mc_alive_diff": (a1 - a2).mean(), "mc_hp_diff": (h1 - h2).mean(),
            "mc_turns": turns.mean(), "mc_turns_std": turns.std(), "mc_capped": capped.mean(),
            "mc_p1_alive": a1.mean(), "mc_p2_alive": a2.mean(), "mc_win_margin": ((a1 - a2) * (2 * win - 1)).mean(),
            "mc_win_std": win.std(), "mc_hp_diff_p10": np.percentile(h1 - h2, 10),
            "mc_hp_diff_p90": np.percentile(h1 - h2, 90), "mc_turns_left_median": np.median(turns)}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--n-sims", type=int, default=100)  # valore usato per le feature dello stack finale
    ap.add_argument("--limit", type=int, default=0, help="solo le prime N battaglie di train (debug)")
    ap.add_argument("--clean", action="store_true", help="usa le timeline pulite dalle mosse persistenti")
    ap.add_argument("--n-jobs", type=int, default=0, help="processi paralleli (default: tutti i core)")
    args = ap.parse_args()
    configure(args.n_jobs)
    print(describe())
    train = load_jsonl(os.path.join(DATA, "train.jsonl"))
    test = load_jsonl(os.path.join(DATA, "test.jsonl"))
    for b in test:
        b["player_won"] = -1
    battles = train[:args.limit] if args.limit else train + test
    # snapshot dei battle di train per l'augmentation dello stacking (stessi turni di stacking.AUG_TURNS)
    snaps = [] if args.limit else [{**b, "battle_timeline": b["battle_timeline"][:t]} for t in SNAP_TURNS for b in train]
    snap_turn = [30] * len(battles) + [len(b["battle_timeline"]) for b in snaps]
    battles = battles + snaps
    t0 = time.time()
    g = build_globals(args.clean)
    rows = pool_map(simulate_battle, [(b, args.n_sims) for b in battles], _set_globals, (g,), chunksize=50)
    df = pd.DataFrame(rows)
    df.insert(0, "battle_id", [b["battle_id"] for b in battles])
    df.insert(1, "is_test", [int(b["player_won"] == -1) for b in battles])
    df.insert(2, "snap_turn", snap_turn)
    y = np.array([b["player_won"] for b in battles])
    trm = (df.is_test.values == 0) & (df.snap_turn.values == 30)
    print(f"{len(df)} battaglie, {args.n_sims} sim ciascuna, {time.time() - t0:.0f}s")
    print(f"accuracy di mc_win > 0.5 sul train: {((df.mc_win.values[trm] > 0.5) == y[trm]).mean():.4f}")
    print(df[trm].describe().T[["mean", "std"]])
    if not args.limit:
        os.makedirs(OUT, exist_ok=True)
        df.to_pickle(os.path.join(OUT, "mc_features.pkl"))


if __name__ == "__main__":
    main()

**`src/recursive.py`**: Recursive model: LightGBM turn-to-turn transition model and roll-outs (RNN-like, no neural nets).

In [ ]:
%%writefile src/recursive.py
"""Modello ricorsivo "stile RNN" con ML classico.

1. Un modello di transizione impara l'esito di un singolo turno:
   (stato_t, memoria_t) -> (delta HP p1, delta HP p2, P(KO p1), P(KO p2)).
   La memoria è una media esponenziale dei delta passati (l'analogo dell'hidden state
   di una RNN). Non usa le etichette, quindi si allena anche sulle timeline di test.
2. Dal turno 30 il modello viene applicato ricorsivamente alle proprie uscite (con
   residui campionati) fino a che un team è esausto: si ricostruisce il resto della battaglia.
3. Le statistiche dei rollout diventano feature per lo stacking.

Uso: python src/recursive.py [--n-rollouts 30] [--min-turn 18]  ->  outputs/rec_features.pkl
"""
import argparse
import os
import sys
import time

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import r2_score, roc_auc_score

sys.path.insert(0, os.path.dirname(__file__))
from compute import configure, describe, n_jobs
from features import load_jsonl

ROOT = os.path.join(os.path.dirname(__file__), "..")
DATA = os.environ.get("POKEMON_DATA", os.path.join(ROOT, "data"))  # su Kaggle: /kaggle/input/...
OUT = os.environ.get("POKEMON_OUT", os.path.join(ROOT, "outputs"))
EMA = 0.8
MAX_TURNS = 200
SNAP_TURNS = (24, 26, 28)
STATE = ["hp1", "hp2", "alive1", "alive2", "slp1", "slp2", "par1", "par2", "psn1", "psn2", "spk1", "spk2",
         "ema_dhp1", "ema_dhp2", "ema_f1", "ema_f2"]
STATIC = ["p1_bst", "p1_spe", "p1_def", "p1_spd"]


def battle_sequence(b):
    """Sequenza degli stati aggregati turno per turno + contesto statico."""
    team = b["p1_team_details"]
    static = [np.mean([sum(p["base_" + s] for s in ("hp", "atk", "def", "spa", "spd", "spe")) for p in team]),
              np.mean([p["base_spe"] for p in team]), np.mean([p["base_def"] for p in team]),
              np.mean([p["base_spd"] for p in team])]
    hp = {"p1": {p["name"]: 1.0 for p in team}, "p2": {}}
    st = {"p1": {}, "p2": {}}
    spikes = {"p1": 0, "p2": 0}
    rows = []
    for t in b["battle_timeline"]:
        for s, o in (("p1", "p2"), ("p2", "p1")):
            ps = t[s + "_pokemon_state"]
            hp[s][ps["name"]], st[s][ps["name"]] = ps["hp_pct"], ps["status"]
            m = t[s + "_move_details"]
            if m and m["name"] == "spikes":
                spikes[o] = 1
            elif m and m["name"] == "rapidspin":
                spikes[s] = 0
        r = []
        for s in ("p1", "p2"):
            unseen = max(6 - len(hp[s]), 0)
            r.append(sum(hp[s].values()) + unseen)
        for s in ("p1", "p2"):
            r.append(6 - sum(1 for h in hp[s].values() if h == 0))
        for code in (("slp",), ("par",), ("psn", "tox")):
            for s in ("p1", "p2"):
                r.append(sum(1 for n, x in st[s].items() if x in code and hp[s][n] > 0))
        r += [spikes["p1"], spikes["p2"]]
        rows.append(r)
    return np.array(rows, dtype=float), np.array(static)


def add_memory(seq):
    """Aggiunge la memoria (EMA dei delta di HP e dei KO) a ogni turno."""
    mem = np.zeros((len(seq), 4))
    h = np.zeros(4)
    for t in range(1, len(seq)):
        d = np.array([seq[t, 0] - seq[t - 1, 0], seq[t, 1] - seq[t - 1, 1],
                      seq[t - 1, 2] - seq[t, 2], seq[t - 1, 3] - seq[t, 3]])
        h = EMA * h + (1 - EMA) * d
        mem[t] = h
    return np.hstack([seq, mem])


def last_state(b):
    """Stato (con memoria) all'ultimo turno della timeline, punto di partenza dei rollout."""
    seq, static = battle_sequence(b)
    return np.concatenate([add_memory(seq)[-1], static])


def build_transitions(battles, min_turn=1):
    """Transizioni turno -> turno. Esclude i turni di rimpiazzo dopo un KO (nessuno agisce
    davvero) e, con min_turn, i primi turni della battaglia."""
    X, Y, last = [], [], []
    for b in battles:
        seq, static = battle_sequence(b)
        S = add_memory(seq)
        last.append(np.concatenate([S[-1], static]))
        for t in range(len(S) - 1):
            if t + 1 < min_turn:
                continue
            if t > 0 and (S[t, 2] < S[t - 1, 2] or S[t, 3] < S[t - 1, 3]):
                continue  # KO nel turno t: t -> t+1 è il turno di rimpiazzo
            X.append(np.concatenate([S[t], static]))
            Y.append([S[t + 1, 0] - S[t, 0], S[t + 1, 1] - S[t, 1],
                      float(S[t + 1, 2] < S[t, 2]), float(S[t + 1, 3] < S[t, 3])])
    return np.array(X), np.array(Y), np.array(last)


def fit_transition(X, Y, n_trees=100):
    """Quattro modelli: due regressori sui delta HP, due classificatori sui KO."""
    # learning rate scalato col numero di alberi: meno alberi = rollout più veloci
    params = dict(n_estimators=n_trees, learning_rate=15 / n_trees, num_leaves=31, min_child_samples=100,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=42, verbose=-1,
                  n_jobs=n_jobs())
    n = len(X)
    rng = np.random.default_rng(0)
    va = rng.random(n) < 0.1
    models = []
    for k in range(4):
        cls = lgb.LGBMClassifier if k >= 2 else lgb.LGBMRegressor
        m = cls(**params).fit(X[~va], Y[~va, k])
        if k >= 2:
            print(f"  P(KO {'p1' if k == 2 else 'p2'}) AUC validation: {roc_auc_score(Y[va, k], m.predict_proba(X[va])[:, 1]):.3f}")
        else:
            print(f"  delta HP {'p1' if k == 0 else 'p2'} R2 validation: {r2_score(Y[va, k], m.predict(X[va])):.3f}")
        models.append(m)
    residuals = [Y[:, k] - models[k].predict(X) for k in (0, 1)]
    return models, residuals


def rollout(models, residuals, start, n_rollouts, seed=0):
    """Applica ricorsivamente il modello di transizione da turno 30 fino alla fine."""
    rng = np.random.default_rng(seed)
    n_b = len(start)
    S = np.repeat(start, n_rollouts, axis=0)
    idx = np.repeat(np.arange(n_b), n_rollouts)
    done = np.zeros(len(S), bool)
    win = np.zeros(len(S))
    turns = np.zeros(len(S))
    snap = {}
    for step in range(1, MAX_TURNS + 1):
        act = np.where(~done)[0]
        if len(act) == 0:
            break
        X = S[act]
        dhp1 = models[0].predict(X) + rng.choice(residuals[0], len(act))
        dhp2 = models[1].predict(X) + rng.choice(residuals[1], len(act))
        f1 = rng.random(len(act)) < models[2].predict_proba(X)[:, 1]
        f2 = rng.random(len(act)) < models[3].predict_proba(X)[:, 1]
        new = X.copy()
        new[:, 2] -= f1
        new[:, 3] -= f2
        new[:, 0] = np.clip(X[:, 0] + dhp1, 0, new[:, 2])
        new[:, 1] = np.clip(X[:, 1] + dhp2, 0, new[:, 3])
        # i dormienti si svegliano con probabilità 1/3 a turno; status persi coi KO
        for c in (4, 5):
            new[:, c] = np.maximum(0, new[:, c] - (rng.random(len(act)) < new[:, c] / 3))
        for c, a in ((4, 2), (5, 3), (6, 2), (7, 3), (8, 2), (9, 3)):
            new[:, c] = np.minimum(new[:, c], new[:, a])
        d = np.column_stack([new[:, 0] - X[:, 0], new[:, 1] - X[:, 1], f1, f2])
        new[:, 12:16] = EMA * X[:, 12:16] + (1 - EMA) * d
        S[act] = new
        end = (new[:, 2] <= 0) | (new[:, 3] <= 0)
        ended = act[end]
        win[ended] = (new[end, 3] <= 0) & (new[end, 2] > 0)
        turns[ended] = step
        done[ended] = True
        if step in (10, 30):
            snap[step] = S[:, [0, 1, 2, 3]].copy()
    open_ = ~done
    win[open_] = S[open_, 0] > S[open_, 1]
    turns[open_] = MAX_TURNS
    agg = lambda v: pd.Series(v).groupby(idx).mean().values
    out = {"rec_win": agg(win), "rec_turns": agg(turns), "rec_capped": agg(open_.astype(float)),
           "rec_alive_diff_end": agg(S[:, 2] - S[:, 3]), "rec_hp_diff_end": agg(S[:, 0] - S[:, 1])}
    for step, v in snap.items():
        out[f"rec_hp_diff_{step}"] = agg(v[:, 0] - v[:, 1])
        out[f"rec_alive_diff_{step}"] = agg(v[:, 2] - v[:, 3])
    return pd.DataFrame(out)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--n-rollouts", type=int, default=30)
    ap.add_argument("--min-turn", type=int, default=1, help="allena solo sulle transizioni dal turno N in poi")
    ap.add_argument("--trees", type=int, default=100, help="alberi per modello di transizione")
    ap.add_argument("--n-jobs", type=int, default=0, help="thread (default: tutti i core)")
    ap.add_argument("--out", default="rec_features.pkl")
    args = ap.parse_args()
    configure(args.n_jobs)
    print(describe())
    t0 = time.time()
    train = load_jsonl(os.path.join(DATA, "train.jsonl"))
    test = load_jsonl(os.path.join(DATA, "test.jsonl"))
    battles = train + test
    X, Y, last = build_transitions(battles, args.min_turn)
    print(f"{len(X)} transizioni da {len(battles)} battaglie ({time.time() - t0:.0f}s)")
    models, residuals = fit_transition(X, Y, args.trees)
    # anche gli snapshot dei battle di train, per l'augmentation dello stacking
    snap_turns = [t for t in SNAP_TURNS for _ in train]
    snap_last = np.array([last_state({**b, "battle_timeline": b["battle_timeline"][:t]})
                          for t in SNAP_TURNS for b in train])
    df = rollout(models, residuals, np.vstack([last, snap_last]), args.n_rollouts)
    df.insert(0, "battle_id", [b["battle_id"] for b in battles] + [b["battle_id"] for _ in SNAP_TURNS for b in train])
    df.insert(1, "is_test", [0] * len(train) + [1] * len(test) + [0] * len(snap_last))
    df.insert(2, "snap_turn", [30] * len(battles) + snap_turns)
    y = np.array([b["player_won"] for b in train])
    trm = (df.is_test.values == 0) & (df.snap_turn.values == 30)
    print(f"rollout completati ({time.time() - t0:.0f}s)")
    print(f"accuracy di rec_win > 0.5 sul train: {((df.rec_win.values[trm] > 0.5) == y).mean():.4f}")
    print(df[trm].describe().T[["mean", "std"]])
    os.makedirs(OUT, exist_ok=True)
    df.to_pickle(os.path.join(OUT, args.out))


if __name__ == "__main__":
    main()

**`src/stacking.py`**: Base model zoo, out-of-fold predictions with snapshot augmentation, duplicate override.

In [ ]:
%%writefile src/stacking.py
"""Stacking: base models eterogenei (alberi, lineari, kernel, generativi) + meta-modello.

Le predizioni out-of-fold (OOF) di ogni base model sono calcolate con k-fold stratificato
e salvate in outputs/oof/<nome>.npz, così si possono aggiungere modelli senza ricalcolare tutto.

Uso:
    python src/stacking.py --models lgbm,xgb,gmm   # calcola (o riusa) le OOF
    python src/stacking.py --meta                  # valuta i meta-modelli e scrive la submission
"""
import argparse
import os
import sys
import time

import numpy as np
import pandas as pd
from scipy.special import logit
from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin, clone
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import QuantileTransformer, StandardScaler
from sklearn.svm import SVC
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

sys.path.insert(0, os.path.dirname(__file__))
from compute import catboost_params, configure, describe, fold_map, fold_plan, n_jobs, xgb_params
from train import LGB_PARAMS, OUT, SEED, add_bags, get_features

N_FOLDS = 5    # sovrascrivibile con --folds
N_REPEATS = 1  # sovrascrivibile con --repeats (k-fold ripetuto con split diversi, OOF mediate)
OOF_DIR = os.path.join(OUT, "oof")


class GMMClassifier(BaseEstimator, ClassifierMixin):
    """Classificatore generativo: una GaussianMixture a covarianza piena per classe.

    La covarianza piena modella la distribuzione congiunta delle feature (non le
    tratta come indipendenti come farebbe Naive Bayes); P(y|x) via teorema di Bayes.
    """

    def __init__(self, n_components=3, covariance_type="full", reg_covar=1e-3, random_state=SEED):
        self.n_components = n_components
        self.covariance_type = covariance_type
        self.reg_covar = reg_covar
        self.random_state = random_state

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.gmms_ = [GaussianMixture(self.n_components, covariance_type=self.covariance_type,
                                      reg_covar=self.reg_covar, n_init=2, random_state=self.random_state)
                      .fit(X[y == c]) for c in self.classes_]
        self.log_prior_ = np.log(np.bincount(y) / len(y))
        return self

    def predict_proba(self, X):
        ll = np.column_stack([g.score_samples(X) for g in self.gmms_]) + self.log_prior_
        ll -= ll.max(axis=1, keepdims=True)
        p = np.exp(ll)
        return p / p.sum(axis=1, keepdims=True)

    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(axis=1)]


class PLSProjection(BaseEstimator, TransformerMixin):
    """Proiezione supervisionata (PLS): poche direzioni massimamente correlate col target.

    A differenza della PCA non scarta le direzioni discriminative; i GMM a valle
    modellano la distribuzione congiunta in questo spazio ridotto.
    """

    def __init__(self, n_components=4):
        self.n_components = n_components

    def fit(self, X, y):
        self.pls_ = PLSRegression(self.n_components).fit(X, y)
        return self

    def transform(self, X):
        return self.pls_.transform(X)


class NoisyFit(BaseEstimator, ClassifierMixin):
    """Data augmentation: aggiunge copie del training set con rumore gaussiano.

    Il rumore è proporzionale alla deviazione standard di ogni feature continua
    (le binarie restano intatte); serve a regolarizzare i modelli che overfittano.
    """

    def __init__(self, estimator, sigma=0.1, n_copies=1, random_state=SEED):
        self.estimator = estimator
        self.sigma = sigma
        self.n_copies = n_copies
        self.random_state = random_state

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        rng = np.random.default_rng(self.random_state)
        std = np.nanstd(X, axis=0)
        cont = np.array([len(np.unique(c[~np.isnan(c)])) > 2 for c in X.T])
        scale = np.where(cont, std * self.sigma, 0.0)
        Xs = [X] + [X + rng.normal(size=X.shape) * scale for _ in range(self.n_copies)]
        self.estimator_ = clone(self.estimator).fit(np.vstack(Xs), np.tile(y, self.n_copies + 1))
        self.classes_ = self.estimator_.classes_
        return self

    def predict_proba(self, X):
        return self.estimator_.predict_proba(np.asarray(X, dtype=float))


# regimi individuati dall'analisi degli errori OOF (src/analyze_errors.py): dove lo stack sbaglia di più
REGIMES = {
    "attrition": lambda X: ((X.p1_n_statused + X.p2_n_statused) >= 3) | ((X.p1_rest_sleep + X.p2_rest_sleep) >= 2),
    "balanced": lambda X: X.diff_hp_sum.abs() < 0.75,
    "setup": lambda X: (X.p1_setup_alive > 0) & (X.p2_setup_alive > 0),
}


class RegimeExpert(BaseEstimator, ClassifierMixin):
    """Esperto di un regime: stesso modello, ma gli esempi fuori dal regime pesano `w_out`."""

    def __init__(self, estimator, regime, w_out=0.2):
        self.estimator = estimator
        self.regime = regime
        self.w_out = w_out

    def fit(self, X, y, sample_weight=None):
        w = np.where(REGIMES[self.regime](X).values, 1.0, self.w_out)
        if sample_weight is not None:
            w = w * sample_weight
        self.estimator_ = clone(self.estimator).fit(X, y, sample_weight=w)
        self.classes_ = self.estimator_.classes_
        return self

    def predict_proba(self, X):
        return self.estimator_.predict_proba(X)


EXTRA_FILES = {"mc": "mc_features.pkl", "rec": "rec_features.pkl", "strat": "strat_features.pkl",
               "lon": "lon_features.pkl"}


AUG_TURNS = (24, 26, 28)  # snapshot usati per l'augmentation (devono essere in train.SNAP_TURNS)
AUG_WEIGHT = 0.5  # Fase 8: 0.25 vince nella stima annidata a 3 modelli ma non nello stack completo (0.6967 vs 0.6981)


def load_data(extra=()):
    """Feature base + (opzionale) feature da simulazione Monte Carlo / modello ricorsivo.

    Restituisce anche gli snapshot dei turni AUG_TURNS (solo train) con le stesse colonne,
    usati dai modelli *_aug dentro i fold di training.
    """
    base, snap = get_features(snapshots=True)
    snap = snap[snap.snap_turn.isin(AUG_TURNS)]
    # bag calcolati insieme, così vocabolario e dummies coincidono
    df = add_bags(pd.concat([base.assign(snap_turn=30), snap], ignore_index=True))
    for name in extra:
        ex = pd.read_pickle(os.path.join(OUT, EXTRA_FILES[name]))
        if "snap_turn" not in ex:
            ex = ex.assign(snap_turn=30)
        df = df.merge(ex, on=["battle_id", "is_test", "snap_turn"], how="left")
    df.columns = [c.replace(" ", "_") for c in df.columns]
    main = df.snap_turn.values == 30
    trm = main & (df.is_test.values == 0)
    tem = main & (df.is_test.values == 1)
    snm = ~main
    y = df.player_won.values[trm]
    feat = df.drop(columns=["battle_id", "is_test", "player_won", "snap_turn"])
    bag_cols = [c for c in feat.columns if "__" in c]
    full = feat.astype(float)
    dense = feat.drop(columns=bag_cols).astype(float).fillna(0)
    part = lambda X, m: X[m].reset_index(drop=True)
    return {
        "full": (part(full, trm), part(full, tem)),
        "dense": (part(dense, trm), part(dense, tem)),
        "snap_full": part(full, snm), "snap_dense": part(dense, snm),
        "train_ids": df.battle_id.values[trm], "snap_ids": df.battle_id.values[snm],
        "snap_y": df.player_won.values[snm],
    }, y, df.battle_id.values[tem]


XGB = dict(n_estimators=700, learning_rate=0.02, max_depth=4, subsample=0.8, colsample_bytree=0.3,
           min_child_weight=5, reg_lambda=5.0, random_state=SEED)
CAT = dict(iterations=1500, learning_rate=0.03, depth=6, l2_leaf_reg=5, random_seed=SEED, verbose=0)


# factory dei modelli multi-thread: device e thread letti da compute al momento della creazione
def _lgb():
    return lgb.LGBMClassifier(**LGB_PARAMS, n_jobs=n_jobs())


def _xgb():
    return xgb.XGBClassifier(**XGB, **xgb_params())


def _cat():
    return CatBoostClassifier(**CAT, **catboost_params())


# nome -> (feature set, factory)
MODELS = {
    # alberi su tutte le feature
    "lgbm": ("full", _lgb),
    "xgb": ("full", _xgb),
    "catboost": ("full", _cat),
    "extratrees": ("full_fill", lambda: ExtraTreesClassifier(800, min_samples_leaf=3, max_features=0.2,
                                                             n_jobs=n_jobs(), random_state=SEED)),
    "rf": ("full_fill", lambda: RandomForestClassifier(800, min_samples_leaf=3, max_features=0.15,
                                                       n_jobs=n_jobs(), random_state=SEED)),
    # alberi senza bag (feature selection: solo le feature di stato)
    "lgbm_dense": ("dense", _lgb),
    "catboost_dense": ("dense", _cat),
    # lineari / kernel / vicinato, con pipeline di normalizzazione
    "logreg": ("full_fill", lambda: make_pipeline(StandardScaler(), LogisticRegression(C=0.02, max_iter=3000))),
    "logreg_dense": ("dense", lambda: make_pipeline(QuantileTransformer(output_distribution="normal", random_state=SEED),
                                                    LogisticRegression(C=0.05, max_iter=3000))),
    "svm": ("dense", lambda: make_pipeline(StandardScaler(), SVC(C=0.5, gamma="scale", probability=True,
                                                                 random_state=SEED))),
    "knn": ("dense", lambda: make_pipeline(StandardScaler(), PCA(30, random_state=SEED),
                                           KNeighborsClassifier(150, weights="distance"))),
    # generativi gaussiani (distribuzione congiunta, covarianza piena)
    "lda": ("dense", lambda: make_pipeline(StandardScaler(), LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"))),
    "qda": ("dense", lambda: make_pipeline(QuantileTransformer(output_distribution="normal", random_state=SEED),
                                           PCA(20, random_state=SEED), QuadraticDiscriminantAnalysis(reg_param=0.1))),
    "gmm": ("dense", lambda: make_pipeline(QuantileTransformer(output_distribution="normal", random_state=SEED),
                                           PLSProjection(4), GMMClassifier(n_components=1, reg_covar=1e-2))),
    "gmm_mix": ("dense", lambda: make_pipeline(QuantileTransformer(output_distribution="normal", random_state=SEED),
                                               PLSProjection(2), GMMClassifier(n_components=6, reg_covar=1e-2))),
    # versioni con rumore gaussiano (regolarizzazione)
    "lgbm_noise": ("dense", lambda: NoisyFit(_lgb(), sigma=0.1, n_copies=2)),
    "xgb_noise": ("full_fill", lambda: NoisyFit(_xgb(), sigma=0.1, n_copies=1)),
    # esperti condizionali (Fase 7): un LGBM per regime
    **{f"lgbm_exp_{r}": ("full", (lambda r=r: RegimeExpert(_lgb(), r))) for r in ("attrition", "balanced", "setup")},
}
# modelli single-thread: i loro fold girano in parallelo (joblib/loky, portabile su ogni OS)
PARALLEL_FOLDS = {"logreg", "logreg_dense", "svm", "knn", "lda", "qda", "gmm", "gmm_mix"}


def get_X(sets, kind):
    if kind == "full_fill":
        tr, te = sets["full"]
        return tr.fillna(0), te.fillna(0)
    return sets[kind]


def get_snap(sets, kind):
    if kind == "full_fill":
        return sets["snap_full"].fillna(0)
    return sets["snap_" + kind]


def run_oof(name, sets, y):
    """OOF di un base model. I nomi *_aug usano il modello base con l'augmentation:
    agli esempi del fold di train si aggiungono i loro snapshot (peso AUG_WEIGHT);
    la validazione resta sul solo turno 30."""
    path = os.path.join(OOF_DIR, f"{name}.npz")
    if os.path.exists(path):
        d = np.load(path)
        return d["oof"], d["test"]
    aug = name.endswith("_aug")
    base = name[:-4] if aug else name
    kind, make = MODELS[base]
    X, Xte = get_X(sets, kind)
    S = get_snap(sets, kind) if aug else None
    splits = [(r, tr, va) for r in range(N_REPEATS)
              for tr, va in StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED + r).split(X, y)]

    def fit_fold(tr, va):
        if aug:
            sm = np.isin(sets["snap_ids"], sets["train_ids"][tr])
            Xa = pd.concat([X.iloc[tr], S[sm]], ignore_index=True)
            ya = np.concatenate([y[tr], sets["snap_y"][sm]])
            wa = np.concatenate([np.ones(len(tr)), np.full(sm.sum(), AUG_WEIGHT)])
            m = make().fit(Xa, ya, sample_weight=wa)
        else:
            m = make().fit(X.iloc[tr], y[tr])
        tr_acc = ((m.predict_proba(X.iloc[tr])[:, 1] > 0.5) == y[tr]).mean()
        return m.predict_proba(X.iloc[va])[:, 1], m.predict_proba(Xte)[:, 1], tr_acc

    t0 = time.time()
    res = fold_map(fit_fold, [(tr, va) for _, tr, va in splits], *fold_plan(base, base in PARALLEL_FOLDS))
    oof, pte = np.zeros(len(y)), np.zeros(len(Xte))
    for (_, _, va), (p_va, p_te, _) in zip(splits, res):
        oof[va] += p_va / N_REPEATS  # con k-fold ripetuto, media delle OOF dei diversi split
        pte += p_te / len(splits)
    tr_acc = [r[2] for r in res]
    acc = ((oof > 0.5) == y).mean()
    print(f"{name:15s} OOF acc {acc:.4f} | train acc {np.mean(tr_acc):.4f} (gap {np.mean(tr_acc) - acc:+.3f}) "
          f"| {time.time() - t0:.0f}s", flush=True)
    np.savez(path, oof=oof, test=pte, train_acc=np.mean(tr_acc))
    return oof, pte


def meta_eval(names, y, n_repeats=5):
    """Valuta i meta-modelli sulle OOF con k-fold ripetuto (seed diverso dai base model)."""
    P = {n: np.load(os.path.join(OOF_DIR, f"{n}.npz")) for n in names}
    Z = np.column_stack([logit(np.clip(P[n]["oof"], 1e-4, 1 - 1e-4)) for n in names])
    Zte = np.column_stack([logit(np.clip(P[n]["test"], 1e-4, 1 - 1e-4)) for n in names])
    # validazione del meta fissa (5x5) per confrontare run con --folds/--repeats diversi
    rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=n_repeats, random_state=SEED + 1)
    metas = {
        "mean": None,
        "logreg_meta": lambda: LogisticRegression(C=1.0, max_iter=2000),
        "logreg_meta_l1": lambda: LogisticRegression(C=0.05, penalty="l1", solver="liblinear"),
        "lgbm_meta": lambda: lgb.LGBMClassifier(n_estimators=200, learning_rate=0.02, num_leaves=4,
                                                min_child_samples=100, random_state=SEED, verbose=-1),
    }
    print("\nbase models (OOF acc):")
    for n in names:
        print(f"  {n:15s} {((P[n]['oof'] > 0.5) == y).mean():.4f}")
    results = {}
    for mname, make in metas.items():
        accs = []
        for tr, va in rskf.split(Z, y):
            if make is None:
                p = Z[va].mean(axis=1)
                accs.append(((p > 0) == y[va]).mean())
            else:
                m = make().fit(Z[tr], y[tr])
                accs.append((m.predict(Z[va]) == y[va]).mean())
        results[mname] = (np.mean(accs), np.std(accs))
        print(f"meta {mname:15s} acc {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    lr = LogisticRegression(C=1.0, max_iter=2000).fit(Z, y)
    print("pesi logreg_meta:", dict(zip(names, np.round(lr.coef_[0], 3))))
    return results, Z, Zte


def duplicate_override(test_ids, pred):
    """Battaglie di test identiche (team, lead e timeline) a una di train ricevono l'etichetta
    di quella: nelle 17 coppie duplicate interne al train l'etichetta coincide sempre."""
    import hashlib, json
    from features import load_jsonl
    from train import DATA
    key = lambda b: hashlib.md5(json.dumps([b["p1_team_details"], b["p2_lead_details"], b["battle_timeline"]],
                                           sort_keys=True).encode()).hexdigest()
    known = {key(b): b["player_won"] for b in load_jsonl(os.path.join(DATA, "train.jsonl"))}
    pos = {bid: i for i, bid in enumerate(test_ids)}
    pred = pred.copy()
    n = 0
    for b in load_jsonl(os.path.join(DATA, "test.jsonl")):
        k = key(b)
        if k in known:
            pred[pos[b["battle_id"]]] = known[k]
            n += 1
    print(f"{n} battaglie di test duplicate di battaglie di train: etichetta copiata")
    return pred


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--models", default="")
    ap.add_argument("--meta", action="store_true")
    ap.add_argument("--no-submit", action="store_true", help="valuta il meta senza scrivere la submission")
    ap.add_argument("--use", default="", help="base model da usare nel meta (default: tutti quelli in cache)")
    ap.add_argument("--extra", default="", help="feature aggiuntive: mc,rec")
    ap.add_argument("--folds", type=int, default=5, help="numero di fold dei base model")
    ap.add_argument("--repeats", type=int, default=1, help="ripetizioni del k-fold con split diversi")
    ap.add_argument("--n-jobs", type=int, default=0, help="processi/thread (default: tutti i core)")
    ap.add_argument("--device", default=None, choices=["auto", "cpu", "cuda"],
                    help="cuda: XGBoost e CatBoost su GPU NVIDIA (Linux/Windows)")
    args = ap.parse_args()
    configure(args.n_jobs, args.device)
    print(describe(), flush=True)
    extra = [e for e in args.extra.split(",") if e]
    global OOF_DIR, N_FOLDS, N_REPEATS
    N_FOLDS, N_REPEATS = args.folds, args.repeats
    tag = "_".join(extra)
    if tag:
        OOF_DIR = os.path.join(OUT, f"oof_{tag}")
    if (N_FOLDS, N_REPEATS) != (5, 1):
        OOF_DIR += f"_k{N_FOLDS}r{N_REPEATS}"
    os.makedirs(OOF_DIR, exist_ok=True)
    sets, y, test_ids = load_data(extra)

    for name in [m for m in args.models.split(",") if m]:
        run_oof(name, sets, y)

    if args.meta:
        names = [m for m in args.use.split(",") if m] or \
                sorted(f[:-4] for f in os.listdir(OOF_DIR) if f.endswith(".npz"))
        results, Z, Zte = meta_eval(names, y)
        best = max((k for k in results if k != "mean"), key=lambda k: results[k][0])
        meta = {"logreg_meta": LogisticRegression(C=1.0, max_iter=2000),
                "logreg_meta_l1": LogisticRegression(C=0.05, penalty="l1", solver="liblinear"),
                "lgbm_meta": lgb.LGBMClassifier(n_estimators=200, learning_rate=0.02, num_leaves=4,
                                                min_child_samples=100, random_state=SEED, verbose=-1)}[best]
        pred = duplicate_override(test_ids, meta.fit(Z, y).predict(Zte))
        if args.no_submit:
            return
        fname = f"submission_stacking{'_' + tag if tag else ''}.csv"
        pd.DataFrame({"battle_id": test_ids, "player_won": pred.astype(int)}).to_csv(
            os.path.join(OUT, fname), index=False)
        print(f"{fname} scritta con {best} (CV {results[best][0]:.4f})")


if __name__ == "__main__":
    main()

**`src/final_model.py`**: Final 10-model stack with a logistic-regression meta-learner.

In [ ]:
%%writefile src/final_model.py
"""Modello finale: lo stack ridotto ai componenti che convengono (report reports/02_report_costi_benefici).

Tiene solo i base model diversi tra loro (boosting con augmentation, ExtraTrees, RF, lineari) e toglie
quelli ridondanti o senza resa misurabile (svm, knn, qda, GMM, varianti con rumore, boosting senza
augmentation, esperti di regime) e le feature strategiche/longitudinali. Tiene invece le feature del
Monte Carlo e del modello ricorsivo (--extra mc,rec, default): con i modelli _aug, calcolate anche sugli
snapshot, valgono +0.44 punti (stack finale 0.6994 con mc,rec contro 0.6937 senza, McNemar p = 0.11).

Confronta lo stack finale con quello di riferimento (22 modelli + mc,rec) sulle stesse battaglie
(differenza appaiata con IC bootstrap e McNemar) e scrive outputs/submission_final.csv.

Uso: python src/final_model.py [--extra mc,rec] [--n-jobs N] [--device auto|cpu|cuda]
"""
import argparse
import os
import sys

import numpy as np
import pandas as pd
from scipy import stats
from scipy.special import logit
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold

sys.path.insert(0, os.path.dirname(__file__))
import stacking
from compute import configure, describe
from train import OUT, SEED

FINAL_MODELS = ["lgbm_aug", "xgb_aug", "lgbm_dense_aug", "catboost_aug", "catboost_dense_aug",
                "extratrees", "rf", "lda", "logreg", "logreg_dense"]
REFERENCE = os.path.join(OUT, "oof_mc_rec")  # stack a 22 modelli, miglior risultato precedente


def logits(d, names, key="oof"):
    return np.column_stack([logit(np.clip(np.load(os.path.join(d, f"{n}.npz"))[key], 1e-4, 1 - 1e-4))
                            for n in names])


def meta_oof(Z, y, repeats=5):
    """Accuracy del meta (logreg sui logit) con 5-fold ripetuto (seed 43, come stacking.meta_eval)
    e predizione OOF mediata sulle ripetizioni, per i confronti appaiati."""
    accs, pm = [], np.zeros(len(y))
    for a, b in RepeatedStratifiedKFold(n_splits=5, n_repeats=repeats, random_state=SEED + 1).split(Z, y):
        p = LogisticRegression(C=1.0, max_iter=2000).fit(Z[a], y[a]).predict_proba(Z[b])[:, 1]
        pm[b] += p / repeats
        accs.append(((p > 0.5) == y[b]).mean())
    return np.mean(accs), np.std(accs), pm


def paired(ok_ref, ok_new, n_boot=5000, seed=0):
    d = ok_new.astype(float) - ok_ref.astype(float)
    bs = d[np.random.default_rng(seed).integers(0, len(d), (n_boot, len(d)))].mean(1)
    b, c = int((ok_ref & ~ok_new).sum()), int((~ok_ref & ok_new).sum())
    p = stats.binomtest(min(b, c), b + c, 0.5).pvalue if b + c else 1.0
    return d.mean(), np.percentile(bs, 2.5), np.percentile(bs, 97.5), p


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--extra", default="mc,rec", help="feature aggiuntive; \"\" per nessuna (-0.44 punti, vedi docstring)")
    ap.add_argument("--out", default="submission_final.csv")
    ap.add_argument("--n-jobs", type=int, default=0)
    ap.add_argument("--device", default=None, choices=["auto", "cpu", "cuda"])
    args = ap.parse_args()
    configure(args.n_jobs, args.device)
    print(describe(), flush=True)
    extra = [e for e in args.extra.split(",") if e]
    # stessa cartella di cache di stacking.py: le OOF di un modello con le stesse feature coincidono
    stacking.OOF_DIR = os.path.join(OUT, "oof_" + "_".join(extra) if extra else "oof_final")
    os.makedirs(stacking.OOF_DIR, exist_ok=True)
    sets, y, test_ids = stacking.load_data(extra)
    for name in FINAL_MODELS:
        stacking.run_oof(name, sets, y)

    Z, Zte = logits(stacking.OOF_DIR, FINAL_MODELS), logits(stacking.OOF_DIR, FINAL_MODELS, "test")
    acc, sd, pm = meta_oof(Z, y)
    print(f"\nstack finale ({len(FINAL_MODELS)} modelli, extra={extra or 'nessuna'}): meta logreg {acc:.4f} ± {sd:.4f}")
    # confronto con lo stack a 22 solo se le sue OOF sono in cache (non su un ambiente nuovo, es. Kaggle)
    ref_names = sorted(f[:-4] for f in os.listdir(REFERENCE) if f.endswith(".npz")) if os.path.isdir(REFERENCE) else []
    if set(ref_names) > set(FINAL_MODELS):
        ref_acc, ref_sd, pr = meta_oof(logits(REFERENCE, ref_names), y)
        d, lo, hi, p = paired((pr > 0.5) == y, (pm > 0.5) == y)
        print(f"riferimento ({len(ref_names)} modelli, mc,rec): {ref_acc:.4f} ± {ref_sd:.4f}")
        print(f"differenza finale - riferimento: {d * 100:+.2f} punti, IC95% [{lo * 100:+.2f}, {hi * 100:+.2f}], McNemar p = {p:.3f}")

    meta = LogisticRegression(C=1.0, max_iter=2000).fit(Z, y)
    print("pesi del meta:", dict(zip(FINAL_MODELS, np.round(meta.coef_[0], 3))))
    pred = stacking.duplicate_override(test_ids, meta.predict(Zte))
    pd.DataFrame({"battle_id": test_ids, "player_won": pred.astype(int)}).to_csv(os.path.join(OUT, args.out), index=False)
    print(f"{args.out} scritta: {len(pred)} righe, p1 vince nel {pred.mean():.1%}")


if __name__ == "__main__":
    main()

### 2. Pipeline

Every step caches its results in `outputs/`.

In [ ]:
def run(script, *args):
    """Runs src/<script> in a subprocess, streams its log and stops the notebook if it fails."""
    t0 = time.time()
    proc = subprocess.Popen([sys.executable, "-u", f"src/{script}", *args], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        if "Warning" not in line and "warnings.warn" not in line:  # hide library warnings
            print(line, end="")
    if proc.wait():
        raise RuntimeError(f"{script} failed with exit code {proc.returncode}")
    print(f"--- {script}: {(time.time() - t0) / 60:.1f} min")

**Monte Carlo features.** 15,000 battles + 30,000 training snapshots × 100 simulations.

In [ ]:
run("simulate.py", "--n-sims", "100")

**Recursive roll-out features.**

In [ ]:
run("recursive.py")

**Features, out-of-fold predictions of the 10 base models, meta-learner.** The battle-state features are built here on first use. The log shows each model's out-of-fold accuracy and the cross-validated accuracy of the meta-learner.

In [ ]:
run("final_model.py", "--out", "submission_final.csv")

### 3. Submission file

In [ ]:
pred = pd.read_csv(os.path.join(WORK_DIR, "submission_final.csv"))
sample = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
submission = sample[["battle_id"]].merge(pred, on="battle_id", how="left")
assert len(submission) == len(sample) and submission["player_won"].notna().all()
submission["player_won"] = submission["player_won"].astype(int)
submission.to_csv(os.path.join(OUT_DIR, "submission.csv"), index=False)
print(f"submission.csv written: {len(submission)} rows, share predicted as player 1 win: "
      f"{submission['player_won'].mean():.3f}")
submission.head()

### 4. Submitting

**From the Kaggle website**

1. Click **Save Version** → **Save & Run All (Commit)**.
2. When the run finishes, open the notebook viewer.
3. Under **Output**, use **Submit to Competition**. This submits `submission.csv`.

**From the command line** (Kaggle CLI, after `kaggle kernels push` has run this notebook)

```bash
kaggle kernels status <user>/pokemon-fds-final
kaggle competitions submit fds-pokemon-battles-prediction-2026 -k <user>/pokemon-fds-final -f submission.csv -v <version> -m "final 10-model stack"
kaggle competitions submissions fds-pokemon-battles-prediction-2026
```